# Libraries

In [ ]:
# !pip install ipympl
%matplotlib widget

In [ ]:
import os
import numpy as np
import itertools
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn import svm
from sklearn.feature_selection import RFE
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, make_scorer

# PLOTTING FUNCTIONS

In [ ]:
def elbow_plot(k_values, f1_scores, model):
    
    # Plot the elbow curve
    plt.plot(k_values, f1_scores, marker='o')
    plt.xlabel('Number of Folds')
    plt.ylabel('F1-score')
    plt.title(f'Elbow Plot - F1-score - {model}')
    plt.show()
    
    # Save the plot as an image file
    output_file = f'/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/f1score_over_folds_graphs/elbowplot_{model}_f1.png'
    # output_file = f'/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/f1score_over_folds_graphs/elbowplot_{model}_f1.png'
    plt.savefig(output_file, bbox_inches='tight')
    
    # Clear the current figure
    plt.close()
 

In [ ]:
def cm_plot(cm_scores, f1_scores, model, total=False):
    if total:
        # Compute the total confusion matrix
        cm_value = np.sum(cm_scores, axis=0)
        title = f'Confusion Matrix - Total F1-score - {model}'
    else:
        # Retrieve the confusion matrix with the highest F1-score
        cm_value = cm_scores[np.argmax(f1_scores)]
        title = f'Confusion Matrix - Highest F1-score - {model}'

    # Define the class labels
    classes = ['Low mental workload', 'High mental workload']

    # Plot the confusion matrix
    plt.figure(figsize=(8, 6))
    plt.imshow(cm_value, interpolation='nearest', cmap='coolwarm')
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')

    # Add the values to the confusion matrix plot
    thresh = cm_value.max() / 2.0
    for i in range(cm_value.shape[0]):
        for j in range(cm_value.shape[1]):
            plt.text(j, i, format(cm_value[i, j], 'd'), ha='center', va='center',
                     color='white' if cm_value[i, j] > thresh else 'black')

    plt.tight_layout()
    plt.show()
    
    # Save the plot as an image file
    output_file = f'/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/confusion_matrices/cmplot_{model}.png'
    # output_file = f'/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/confusion_matrices/cmplot_{model}.png'
    plt.savefig(output_file, bbox_inches='tight')
    
    # Clear the current figure
    plt.close()
    

In [ ]:
def cirkel_diagram(channel_counts, model):
    # Define channel groups
    channel_groups = {
        'P': [channel for channel in channel_counts.keys() if channel.startswith('P')],
        'F': [channel for channel in channel_counts.keys() if channel.startswith('F')],
        'C': [channel for channel in channel_counts.keys() if channel.startswith('C')],
        'O': [channel for channel in channel_counts.keys() if channel.startswith('O')],
        'T': [channel for channel in channel_counts.keys() if channel.startswith('T')]
    }

    # Count the number of channels in each group
    group_counts = {group: 0 for group in channel_groups}
    for channel, count in channel_counts.items():
        for group, channels in channel_groups.items():
            if channel in channels:
                group_counts[group] += count

    # Plotting the pie chart
    labels = group_counts.keys()
    sizes = group_counts.values()
    colors = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99', '#c2c2f0']
    explode = [0.1] * len(group_counts)  # explode all the slices

    plt.pie(sizes, labels=labels, colors=colors, explode=explode, autopct='%1.1f%%', startangle=90)

    plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
    plt.title(f'Brain Area Distribution of chosen power band features - {model}')
    plt.show()
    
    # Save the plot as an image file
    output_file = f'/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/brain_area_distribution_selected_spectral_features/cirkeldiagram_{model}.png'
    # output_file = f'/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/brain_area_distribution_selected_spectral_features/cirkeldiagram_{model}.png'
    plt.savefig(output_file, bbox_inches='tight')
    
    # Clear the current figure
    plt.close()

In [ ]:
def barplot_features(channel_counts, frequency_bands, all_channels, model):

    # Extract unique channels and their counts
    unique_channels = list(channel_counts.keys())

    # Sort unique channels based on custom order
    custom_order = ['F', 'O', 'T', 'C', 'P']
    unique_channels = sorted(unique_channels, key=lambda x: (custom_order.index(x[0]), x))

    # Assign colors to each frequency band
    colors = ['r', 'g', 'b']

    # Plotting the stacked barplot
    x = np.arange(len(all_channels))
    
    fig, ax = plt.subplots(figsize=(8, 6))
    bottom = np.zeros(len(all_channels))

    for i, freq in zip(range(len(frequency_bands)), ['Alpha', 'Beta', 'Theta']):
        values = [channel_counts[channel] if channel in frequency_bands[i] else 0 for channel in all_channels]
        ax.bar(x, values, bottom=bottom, label=f"{freq}", color=colors[i])
        bottom += values
    
    ax.set_xlabel('Channels chosen accross frequency ranges', fontsize=10)
    ax.set_title(f'Chosen spectral features by {model}', fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels(all_channels, rotation=45, ha='right', fontsize=8)
    ax.yaxis.set_ticks([])  # Remove ticks on y-axis
    ax.legend()

    plt.show()
    
    # Save the plot as an image file
    output_file = f'/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/barplot_for_selected_spectral_features/barplotfeatures_{model}.png'
    # output_file = f'/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/barplot_for_selected_spectral_features/barplotfeatures_{model}.png'
    plt.savefig(output_file, bbox_inches='tight')
    
    # Clear the current figure
    plt.close()

In [ ]:
def barplot_PLVfeatures(channel_counts, frequency_bands, all_channels, model):

    # Extract unique channels and their counts
    unique_channels = list(channel_counts.keys())
    
    if len(unique_channels) > 1:
        # Sort unique channels based on custom order
        custom_order = ['a', 'b', 't']
        unique_channels = sorted(unique_channels, key=lambda x: (custom_order.index(x[0]), x))

    # Assign colors to each frequency band
    colors = ['r', 'g', 'b']

    # Plotting the stacked barplot
    x = np.arange(len(all_channels))

    fig, ax = plt.subplots(figsize=(8, 6))
    bottom = np.zeros(len(all_channels))

    for i, freq in zip(range(len(frequency_bands)), ['Alpha', 'Beta', 'Theta']):
        values = [channel_counts[channel] if channel in frequency_bands[i] else 0 for channel in all_channels]
        ax.bar(x, values, bottom=bottom, label=f"{freq}", color=colors[i])
        bottom += values

    # Adjust the position and size of the axes
    ax.set_position([0.1, 0.1, 0.9, 0.3])  # Adjust the values as desired 
    ax.set_xlabel('PLV value per channel and the frequency ranges', fontsize=10)
    ax.set_title(f'Chosen PLV features by {model}', fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels(all_channels, rotation=45, ha='right', fontsize=8)
    ax.yaxis.set_ticks([])  # Remove ticks on y-axis

    plt.show()
    
    # Save the plot as an image file
    output_file = f'/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/barplots_for_selected_PLV_features/barplotPLVfeatures_{model}.png'
    # output_file = f'/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/barplots_for_selected_PLV_features/barplotPLVfeatures_{model}.png'
    plt.savefig(output_file, bbox_inches='tight')
    
    # Clear the current figure
    plt.close()

# Frontal-Parietal PLV value calculation

In [ ]:
def frontal_parietal_PLV(data):
    '''
    Calculates the connectivity between the frontal and parietal areas in the brain.
    
    Parameters:
        data (array-like): Connectivity data between each channel.
    
    Computation: 
        - mean connectivity for each channel based on it's connectivity values with all opposite brain area channels.
        - mean connectivity for each brain area
        - mean connectivity between frontal and parietal brain areas.
    
    Returns:
        numpy.ndarray: Frontal-parietal connectivity values for each frequency band (alpha, beta, theta).
    '''
        
    # alpha inter repetitive plv mean values
    F7 = np.mean(data[4:9])
    F3 = np.mean(data[12:17])
    Fz = np.mean(data[19:24])
    F4 = np.mean(data[25:30])
    F8 = np.mean(data[30:35])
    P3 = np.mean(np.concatenate((data[4:5], data[12:13], data[19:20], data[25:26], data[30:31])))
    Pz = np.mean(np.concatenate((data[5:6], data[13:14], data[20:21], data[26:27], data[31:32])))
    P4 = np.mean(np.concatenate((data[6:7], data[14:15], data[21:22], data[27:28], data[32:33])))
    PO7 = np.mean(np.concatenate((data[7:8], data[15:16], data[22:23], data[28:29], data[33:34])))
    PO8 = np.mean(np.concatenate((data[8:9], data[16:17], data[23:24], data[29:30], data[34:35])))
    Frontal_alpha = np.mean(np.concatenate(([F7], [F3], [Fz], [F4], [F8])))
    Parietal_alpha = np.mean(np.concatenate(([P3], [Pz], [P4], [PO7], [PO8])))
    PLV_alpha = np.mean(np.concatenate(([Frontal_alpha], [Parietal_alpha])))
    # print('Alpha - Inter-repetitive higher-order mean: ', PLV_alpha)

    # beta inter repetitive plv mean values
    F7 = np.mean(data[4+45:9+45])
    F3 = np.mean(data[12+45:17+45])
    Fz = np.mean(data[19+45:24+45])
    F4 = np.mean(data[25+45:30+45])
    F8 = np.mean(data[30+45:35+45])
    P3 = np.mean(np.concatenate((data[4+45:5+45], data[12+45:13+45], data[19+45:20+45], data[25+45:26+45], data[30+45:31+45])))
    Pz = np.mean(np.concatenate((data[5+45:6+45], data[13+45:14+45], data[20+45:21+45], data[26+45:27+45], data[31+45:32+45])))
    P4 = np.mean(np.concatenate((data[6+45:7+45], data[14+45:15+45], data[21+45:22+45], data[27+45:28+45], data[32+45:33+45])))
    PO7 = np.mean(np.concatenate((data[7+45:8+45], data[15+45:16+45], data[22+45:23+45], data[28+45:29+45], data[33+45:34+45])))
    PO8 = np.mean(np.concatenate((data[8+45:9+45], data[16+45:17+45], data[23+45:24+45], data[29+45:30+45], data[34+45:35+45])))
    Frontal_beta = np.mean(np.concatenate(([F7], [F3], [Fz], [F4], [F8])))
    Parietal_beta = np.mean(np.concatenate(([P3], [Pz], [P4], [PO7], [PO8])))
    PLV_beta = np.mean(np.concatenate(([Frontal_beta], [Parietal_beta])))
    # print('Beta - Inter-repetitive higher-order mean: ', PLV_beta)

    # theta inter repetitive plv mean values
    F7 = np.mean(data[4+90:9+90])
    F3 = np.mean(data[12+90:17+90])
    Fz = np.mean(data[19+90:24+90])
    F4 = np.mean(data[25+90:30+90])
    F8 = np.mean(data[30+90:35+90])
    P3 = np.mean(np.concatenate((data[4+90:5+90], data[12+90:13+90], data[19+90:20+90], data[25+90:26+90], data[30+90:31+90])))
    Pz = np.mean(np.concatenate((data[5+90:6+90], data[13+90:14+90], data[20+90:21+90], data[26+90:27+90], data[31+90:32+90])))
    P4 = np.mean(np.concatenate((data[6+90:7+90], data[14+90:15+90], data[21+90:22+90], data[27+90:28+90], data[32+90:33+90])))
    PO7 = np.mean(np.concatenate((data[7+90:8+90], data[15+90:16+90], data[22+90:23+90], data[28+90:29+90], data[33+90:34+90])))
    PO8 = np.mean(np.concatenate((data[8+90:9+90], data[16+90:17+90], data[23+90:24+90], data[29+90:30+90], data[34+90:35+90])))
    Frontal_theta = np.mean(np.concatenate(([F7], [F3], [Fz], [F4], [F8])))
    Parietal_theta = np.mean(np.concatenate(([P3], [Pz], [P4], [PO7], [PO8])))
    PLV_theta = np.mean(np.concatenate(([Frontal_theta], [Parietal_theta])))
    # print('Theta - Inter-repetitive higher-order mean: ', PLV_theta)
    
    data_x = np.concatenate((np.array([PLV_alpha]), np.array([PLV_beta]), np.array([PLV_theta])))
    
    return data_x


In [ ]:
def frontal_parietal_PLVs(data):
    '''
    Calculates the connectivity between the frontal and parietal areas in the brain.
    
    Parameters:
        data (array-like): Connectivity data between each channel.
    
    Computation: 
        - mean connectivity for each channel based on it's connectivity values with all opposite brain area channels.
        - mean connectivity for each brain area
        - mean connectivity between frontal and parietal brain areas.
    
    Returns:
        numpy.ndarray: Frontal-parietal connectivity values for each frequency band (alpha, beta, theta) and for each channel and it's connectivity with the other side.
    '''
        
    # alpha inter repetitive plv mean values
    F7 = np.mean(data[4:9])
    F3 = np.mean(data[12:17])
    Fz = np.mean(data[19:24])
    F4 = np.mean(data[25:30])
    F8 = np.mean(data[30:35])
    P3 = np.mean(np.concatenate((data[4:5], data[12:13], data[19:20], data[25:26], data[30:31])))
    Pz = np.mean(np.concatenate((data[5:6], data[13:14], data[20:21], data[26:27], data[31:32])))
    P4 = np.mean(np.concatenate((data[6:7], data[14:15], data[21:22], data[27:28], data[32:33])))
    PO7 = np.mean(np.concatenate((data[7:8], data[15:16], data[22:23], data[28:29], data[33:34])))
    PO8 = np.mean(np.concatenate((data[8:9], data[16:17], data[23:24], data[29:30], data[34:35])))
    PLVs_alpha = np.concatenate(([F7], [F3], [Fz], [F4], [F8], [P3], [Pz], [P4], [PO7], [PO8]))
    # print('Alpha - Inter-repetitive means: ', PLVs_alpha)

    # beta inter repetitive plv mean values
    F7 = np.mean(data[4+45:9+45])
    F3 = np.mean(data[12+45:17+45])
    Fz = np.mean(data[19+45:24+45])
    F4 = np.mean(data[25+45:30+45])
    F8 = np.mean(data[30+45:35+45])
    P3 = np.mean(np.concatenate((data[4+45:5+45], data[12+45:13+45], data[19+45:20+45], data[25+45:26+45], data[30+45:31+45])))
    Pz = np.mean(np.concatenate((data[5+45:6+45], data[13+45:14+45], data[20+45:21+45], data[26+45:27+45], data[31+45:32+45])))
    P4 = np.mean(np.concatenate((data[6+45:7+45], data[14+45:15+45], data[21+45:22+45], data[27+45:28+45], data[32+45:33+45])))
    PO7 = np.mean(np.concatenate((data[7+45:8+45], data[15+45:16+45], data[22+45:23+45], data[28+45:29+45], data[33+45:34+45])))
    PO8 = np.mean(np.concatenate((data[8+45:9+45], data[16+45:17+45], data[23+45:24+45], data[29+45:30+45], data[34+45:35+45])))
    PLVs_beta = np.concatenate(([F7], [F3], [Fz], [F4], [F8], [P3], [Pz], [P4], [PO7], [PO8]))
    # print('Beta - Inter-repetitive means: ', PLVs_beta)

    # theta inter repetitive plv mean values
    F7 = np.mean(data[4+90:9+90])
    F3 = np.mean(data[12+90:17+90])
    Fz = np.mean(data[19+90:24+90])
    F4 = np.mean(data[25+90:30+90])
    F8 = np.mean(data[30+90:35+90])
    P3 = np.mean(np.concatenate((data[4+90:5+90], data[12+90:13+90], data[19+90:20+90], data[25+90:26+90], data[30+90:31+90])))
    Pz = np.mean(np.concatenate((data[5+90:6+90], data[13+90:14+90], data[20+90:21+90], data[26+90:27+90], data[31+90:32+90])))
    P4 = np.mean(np.concatenate((data[6+90:7+90], data[14+90:15+90], data[21+90:22+90], data[27+90:28+90], data[32+90:33+90])))
    PO7 = np.mean(np.concatenate((data[7+90:8+90], data[15+90:16+90], data[22+90:23+90], data[28+90:29+90], data[33+90:34+90])))
    PO8 = np.mean(np.concatenate((data[8+90:9+90], data[16+90:17+90], data[23+90:24+90], data[29+90:30+90], data[34+90:35+90])))
    PLVs_theta = np.concatenate(([F7], [F3], [Fz], [F4], [F8], [P3], [Pz], [P4], [PO7], [PO8]))
    # print('Theta - Inter-repetitive means: ', PLVs_theta)
    
    data_x = np.concatenate((PLVs_alpha, PLVs_beta, PLVs_theta))
    
    return data_x


# Raw model - all spectral features

In [ ]:
# loading data 

directory = '/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'
# directory = '/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'

concatenated_X, concatenated_Y = [], []

for subdir in sorted(os.listdir(directory))[1:]: # index from 1 because Mac has .DS_Store file
    for file in sorted(os.listdir(os.path.join(directory, subdir))):
        data = np.load(os.path.join(directory, subdir, file))
        X = data['X'][135:] # extract all spectral features
        Y = data['Y']
        concatenated_X.append(X)
        concatenated_Y.append(Y)

concatenated_Y = np.concatenate(concatenated_Y)
    

In [ ]:
# SVM

X = concatenated_X
Y = concatenated_Y

# Create an SVM classifier
classifier = svm.SVC(random_state=42)

# Set the hyperparameters to tune
parameters = {'C': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.2, 1.5, 1.7, 2, 2.2, 2.5, 2.7, 3, 3.5, 4, 4.5, 5, 6, 7, 8, 9, 10, 12, 15, 17, 20, 22, 25, 27, 30, 35, 40, 45, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': ['auto', 'scale', 0.000001, 0.000002, 0.000003, 0.000004, 0.000005, 0.00001, 0.00002, 0.00003, 0.00004, 0.00005, 0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.001, 0.002, 0.003, 0.004, 0.005, 0.01, 0.02, 0.03, 0.04, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.6, 0.8, 1, 2, 3, 5, 6, 8, 10, 12, 15, 17, 20, 22, 25, 27, 30]}
# parameters = {'C': [0.1, 0.3, 1, 5, 10, 16, 30, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': [0.00001, 0.0005, 0.005, 0.01, 0.05, 0.15, 1, 3, 6]}
# Define the scoring metric
scoring = make_scorer(f1_score, average='macro')

# Set the number of folds for cross-validation
num_folds = 10

# Create the Stratified k-fold object
skf = StratifiedKFold(n_splits=num_folds, shuffle=False)

f1_scores, accuracy_scores, precision_scores, recall_scores, cm_scores, best_parameters_list = [],[],[],[],[],[]
k_values = list(range(1, num_folds + 1))

# Perform stratified cross-validation
for k, (train_index, test_index) in enumerate(skf.split(X, Y), start=1):
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    Y_train, Y_test = Y[train_index], Y[test_index]

    # Use grid search to tune the hyperparameters with F1-score as the scoring metric
    grid_search = GridSearchCV(classifier, parameters, scoring=scoring)
    grid_search.fit(X_train, Y_train)

    # Get the best hyperparameters
    best_parameters = grid_search.best_params_

    # Train the classifier using the best hyperparameters
    classifier = svm.SVC(kernel=best_parameters['kernel'], C=best_parameters['C'], gamma=best_parameters['gamma'])
    classifier.fit(X_train, Y_train)

    # Make predictions on the test set
    Y_pred = classifier.predict(X_test)

    # Calculater performance metrics and retrieve best parameter set
    best_parameters_list.append(best_parameters) 
    f1_scores.append(f1_score(Y_test, Y_pred, average='macro'))
    accuracy_scores.append(accuracy_score(Y_test, Y_pred))
    precision_scores.append(precision_score(Y_test, Y_pred, average='macro', zero_division=1))
    recall_scores.append(recall_score(Y_test, Y_pred, average='macro'))
    cm_scores.append(confusion_matrix(Y_test, Y_pred))
    
    # Print the progress
    f1 = f1_score(Y_test, Y_pred, average='macro')
    print(f'Fold {k} - F1-score: {round(f1, 2)} - Best Parameters: {best_parameters}')
    
# print the models performance metrices and its best parameter set
print(' ')
print('Mean F1-score:', f'{round(np.mean(f1_scores), 2)}.', 'Std F1-score:', f'{round(np.std(f1_scores), 2)}.','Max F1-score:', round(np.max(f1_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(f1_scores)]}')
print('Mean Accuracy-score:', f'{round(np.mean(accuracy_scores), 2)}.', 'Std Accuracy-score:', f'{round(np.std(accuracy_scores), 2)}.', 'Max Accuracy-score:', round(np.max(accuracy_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(accuracy_scores)]}')
print('Mean Precision-score:', f'{round(np.mean(precision_scores), 2)}.', 'Std Precision-score:', f'{round(np.std(precision_scores), 2)}.', 'Max Precision-score:', round(np.max(precision_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(precision_scores)]}')
print('Mean Recall-score:', f'{round(np.mean(recall_scores), 2)}.', 'Std Recall-score:', f'{round(np.std(recall_scores), 2)}.', 'Max Recall-score:', round(np.max(recall_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(recall_scores)]}')

elbow_plot(k_values, f1_scores, model='Raw')
cm_plot(cm_scores, f1_scores, model='Raw', total=True)

raw_model_f1 = f1_scores
raw_model_param = best_parameters_list[np.argmax(f1_scores)]

In [ ]:
# features selected by the model

node_names_orig = ['F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'Cz', 'T8', 'P3', 'Pz', 'P4', 'PO7', 'PO8', 'Oz']

# Generate all possible combination pairs of nodes
combinations_total = node_names_orig*3

alpha_spec, beta_spec, theta_spec = [], [], []
for i in range(len(combinations_total)):
    if i < 14:
        alpha_spec.append(combinations_total[i])
    elif i >= 14 and i < 28:
        beta_spec.append(combinations_total[i])
    elif i >= 28 and i < 42:
        theta_spec.append(combinations_total[i])
print('Spectrum: ', len(alpha_spec), len(beta_spec), len(theta_spec))

frontal_count, parietal_count = 0, 0
channel_counts = {}
frequency_bands = [alpha_spec, beta_spec, theta_spec]
for frequency_band in frequency_bands:
    for channel in frequency_band:
        # if 'F' in channel:
        #     frontal_count += 1
        # elif 'P' in channel:
        #     parietal_count += 1
        if channel in channel_counts:
            channel_counts[channel] += 1
        else:
            channel_counts[channel] = 1
    
# cirkel_diagram(channel_counts, model='Raw')
# barplot_features(channel_counts, frequency_bands, node_names_orig, model='Raw')

# Baseline - RFE on all spectral features

In [ ]:
# loading data 

directory = '/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'
# directory = '/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'

concatenated_X, concatenated_Y = [], []

for subdir in sorted(os.listdir(directory))[1:]: # index from 1 because Mac has .DS_Store file
    for file in sorted(os.listdir(os.path.join(directory, subdir))):
        data = np.load(os.path.join(directory, subdir, file))
        X = data['X'][135:] # extract all spectral features
        Y = data['Y']
        concatenated_X.append(X)
        concatenated_Y.append(Y)

concatenated_Y = np.concatenate(concatenated_Y)
    

In [ ]:
# SVM + RFE

X = concatenated_X
Y = concatenated_Y

# Create an SVM classifier
classifier = svm.SVC(random_state=42)

# Set the hyperparameters to tune
# parameters = {'C': [0.1, 0.3, 1, 5, 10, 16, 30, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': [0.00001, 0.0005, 0.005, 0.01, 0.05, 0.15, 1, 3, 6]}
parameters = {'C': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.2, 1.5, 1.7, 2, 2.2, 2.5, 2.7, 3, 3.5, 4, 4.5, 5, 6, 7, 8, 9, 10, 12, 15, 17, 20, 22, 25, 27, 30, 35, 40, 45, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': ['auto', 'scale', 0.000001, 0.000002, 0.000003, 0.000004, 0.000005, 0.00001, 0.00002, 0.00003, 0.00004, 0.00005, 0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.001, 0.002, 0.003, 0.004, 0.005, 0.01, 0.02, 0.03, 0.04, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.6, 0.8, 1, 2, 3, 5, 6, 8, 10, 12, 15, 17, 20, 22, 25, 27, 30]}

# Define the scoring metric
scoring = make_scorer(f1_score, average='macro')

# Set the number of folds for cross-validation
num_folds = 10

# Create the Stratified k-fold object
skf = StratifiedKFold(n_splits=num_folds, shuffle=False)

f1_scores, accuracy_scores, precision_scores, recall_scores, cm_scores, best_parameters_list = [],[],[],[],[],[]
k_values = list(range(1, num_folds + 1))

# Perform stratified cross-validation
for k, (train_index, test_index) in enumerate(skf.split(X, Y), start=1):
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    Y_train, Y_test = Y[train_index], Y[test_index]
    
    # Perform feature selection using RFE with LinearSVC as the estimator
    estimator = svm.LinearSVC(max_iter=30000)  # Use LinearSVC as the estimator for feature selection
    selector = RFE(estimator)
    selector.fit(X_train, Y_train)
    X_train_selected = selector.transform(X_train)
    X_test_selected = selector.transform(X_test)
    
    # Get the selected feature indices
    selected_feature_indices = selector.support_
    selected_feature_ranking = selector.ranking_

    # Use grid search to tune the hyperparameters with F1-score as the scoring metric
    grid_search = GridSearchCV(classifier, parameters, scoring=scoring)
    grid_search.fit(X_train_selected, Y_train)

    # Get the best hyperparameters
    best_parameters = grid_search.best_params_

    # Train the classifier using the best hyperparameters
    classifier = svm.SVC(kernel=best_parameters['kernel'], C=best_parameters['C'], gamma=best_parameters['gamma'])
    classifier.fit(X_train_selected, Y_train)

    # Make predictions on the test set
    Y_pred = classifier.predict(X_test_selected)

    # Calculater performance metrics and retrieve best parameter set
    best_parameters_list.append(best_parameters) 
    f1_scores.append(f1_score(Y_test, Y_pred, average='macro'))
    accuracy_scores.append(accuracy_score(Y_test, Y_pred))
    precision_scores.append(precision_score(Y_test, Y_pred, average='macro', zero_division=1))
    recall_scores.append(recall_score(Y_test, Y_pred, average='macro'))
    cm_scores.append(confusion_matrix(Y_test, Y_pred))
    
    # Print the progress
    f1 = f1_score(Y_test, Y_pred, average='macro')
    print(f'Fold {k} - F1-score: {round(f1, 2)} - Best Parameters: {best_parameters}')
    
# print the models performance metrices and its best parameter set
print(' ')
print('Mean F1-score:', f'{round(np.mean(f1_scores), 2)}.', 'Std F1-score:', f'{round(np.std(f1_scores), 2)}.','Max F1-score:', round(np.max(f1_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(f1_scores)]}')
print(' ')
print('Mean Accuracy-score:', f'{round(np.mean(accuracy_scores), 2)}.', 'Std Accuracy-score:', f'{round(np.std(accuracy_scores), 2)}.', 'Max Accuracy-score:', round(np.max(accuracy_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(accuracy_scores)]}')
print(' ')
print('Mean Precision-score:', f'{round(np.mean(precision_scores), 2)}.', 'Std Precision-score:', f'{round(np.std(precision_scores), 2)}.', 'Max Precision-score:', round(np.max(precision_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(precision_scores)]}')
print(' ')
print('Mean Recall-score:', f'{round(np.mean(recall_scores), 2)}.', 'Std Recall-score:', f'{round(np.std(recall_scores), 2)}.', 'Max Recall-score:', round(np.max(recall_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(recall_scores)]}')
print(' ')

elbow_plot(k_values, f1_scores, model='Baseline')
cm_plot(cm_scores, f1_scores, model='Baseline', total=True)

baseline_model_f1 = f1_scores
baseline_model_param = best_parameters_list[np.argmax(f1_scores)]

In [ ]:
# features selected by the model

# Get the indexes where the array is True
indexes_baseline = np.where(selected_feature_indices)[0]

node_names_orig = ['F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'Cz', 'T8', 'P3', 'Pz', 'P4', 'PO7', 'PO8', 'Oz']

# Generate all possible combination pairs of nodes
combinations_total = node_names_orig*3

alpha_spec, beta_spec, theta_spec = [], [], []
for i in indexes_baseline:
    if i < 14:
        alpha_spec.append(combinations_total[i])
    elif i >= 14 and i < 28:
        beta_spec.append(combinations_total[i])
    elif i >= 28 and i < 42:
        theta_spec.append(combinations_total[i])
print('Spectrum: ', len(alpha_spec), len(beta_spec), len(theta_spec))

frontal_count, parietal_count = 0, 0
channel_counts = {}
frequency_bands = [alpha_spec, beta_spec, theta_spec]
for frequency_band in frequency_bands:
    for channel in frequency_band:
        # if 'F' in channel:
        #     frontal_count += 1
        # elif 'P' in channel:
        #     parietal_count += 1
        if channel in channel_counts:
            channel_counts[channel] += 1
        else:
            channel_counts[channel] = 1

cirkel_diagram(channel_counts, model='Baseline')
barplot_features(channel_counts, frequency_bands, node_names_orig, model='Baseline')

chosen_frequency_bands = [alpha_spec, beta_spec, theta_spec]
  

# Model 1 - all spectral features + all 3 frontal-parietal PLV features

In [ ]:
# loading data

directory = '/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'
# directory = '/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'

concatenated_X, concatenated_Y = [], []

for subdir in sorted(os.listdir(directory))[1:]: # index from 1 because Mac has .DS_Store file
    for file in sorted(os.listdir(os.path.join(directory, subdir))):
        data = np.load(os.path.join(directory, subdir, file))
        X_PLV = frontal_parietal_PLV(data['X'][:135]) # 3 PLV features (alpha, beta, theta)
        X_Freq = data['X'][135:] # frequency spectrum features
        X = np.concatenate((np.array(X_PLV), np.array(X_Freq)))
        Y = data['Y']
        concatenated_X.append(X)
        concatenated_Y.append(Y)

concatenated_Y = np.concatenate(concatenated_Y)
    

In [ ]:
# SVM

X = concatenated_X
Y = concatenated_Y

# Create an SVM classifier
classifier = svm.SVC(random_state=42)

# Set the hyperparameters to tune
# parameters = {'C': [0.1, 0.3, 1, 5, 10, 16, 30, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': [0.00001, 0.0005, 0.005, 0.01, 0.05, 0.15, 1, 3, 6]}
parameters = {'C': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.2, 1.5, 1.7, 2, 2.2, 2.5, 2.7, 3, 3.5, 4, 4.5, 5, 6, 7, 8, 9, 10, 12, 15, 17, 20, 22, 25, 27, 30, 35, 40, 45, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': ['auto', 'scale', 0.000001, 0.000002, 0.000003, 0.000004, 0.000005, 0.00001, 0.00002, 0.00003, 0.00004, 0.00005, 0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.001, 0.002, 0.003, 0.004, 0.005, 0.01, 0.02, 0.03, 0.04, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.6, 0.8, 1, 2, 3, 5, 6, 8, 10, 12, 15, 17, 20, 22, 25, 27, 30]}
# Define the scoring metric
scoring = make_scorer(f1_score, average='macro')

# Set the number of folds for cross-validation
num_folds = 10

# Create the Stratified k-fold object
skf = StratifiedKFold(n_splits=num_folds, shuffle=False)

f1_scores, accuracy_scores, precision_scores, recall_scores, cm_scores, best_parameters_list = [],[],[],[],[],[]
k_values = list(range(1, num_folds + 1))

# Perform stratified cross-validation
for k, (train_index, test_index) in enumerate(skf.split(X, Y), start=1):
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    Y_train, Y_test = Y[train_index], Y[test_index]

    # Use grid search to tune the hyperparameters with F1-score as the scoring metric
    grid_search = GridSearchCV(classifier, parameters, scoring=scoring)
    grid_search.fit(X_train, Y_train)

    # Get the best hyperparameters
    best_parameters = grid_search.best_params_

    # Train the classifier using the best hyperparameters
    classifier = svm.SVC(kernel=best_parameters['kernel'], C=best_parameters['C'], gamma=best_parameters['gamma'])
    classifier.fit(X_train, Y_train)

    # Make predictions on the test set
    Y_pred = classifier.predict(X_test)

    # Calculater performance metrics and retrieve best parameter set
    best_parameters_list.append(best_parameters) 
    f1_scores.append(f1_score(Y_test, Y_pred, average='macro'))
    accuracy_scores.append(accuracy_score(Y_test, Y_pred))
    precision_scores.append(precision_score(Y_test, Y_pred, average='macro', zero_division=1))
    recall_scores.append(recall_score(Y_test, Y_pred, average='macro'))
    cm_scores.append(confusion_matrix(Y_test, Y_pred))
    
    # Print the progress
    f1 = f1_score(Y_test, Y_pred, average='macro')
    print(f'Fold {k} - F1-score: {round(f1, 2)} - Best Parameters: {best_parameters}')
    
# print the models performance metrices and its best parameter set
print(' ')
print('Mean F1-score:', f'{round(np.mean(f1_scores), 2)}.', 'Std F1-score:', f'{round(np.std(f1_scores), 2)}.','Max F1-score:', round(np.max(f1_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(f1_scores)]}')
print('Mean Accuracy-score:', f'{round(np.mean(accuracy_scores), 2)}.', 'Std Accuracy-score:', f'{round(np.std(accuracy_scores), 2)}.', 'Max Accuracy-score:', round(np.max(accuracy_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(accuracy_scores)]}')
print('Mean Precision-score:', f'{round(np.mean(precision_scores), 2)}.', 'Std Precision-score:', f'{round(np.std(precision_scores), 2)}.', 'Max Precision-score:', round(np.max(precision_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(precision_scores)]}')
print('Mean Recall-score:', f'{round(np.mean(recall_scores), 2)}.', 'Std Recall-score:', f'{round(np.std(recall_scores), 2)}.', 'Max Recall-score:', round(np.max(recall_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(recall_scores)]}')

elbow_plot(k_values, f1_scores, model='Model1')
cm_plot(cm_scores, f1_scores, model='Model1', total=True)

model1_f1 = f1_scores
model1_param = best_parameters_list[np.argmax(f1_scores)]

In [ ]:
# features selected by the model

node_names_orig = ['F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'Cz', 'T8', 'P3', 'Pz', 'P4', 'PO7', 'PO8', 'Oz']

# Generate all possible combination pairs of nodes
combinations_total = node_names_orig*3

alpha_spec, beta_spec, theta_spec = [], [], []
for i in range(len(combinations_total)):
    if i < 14:
        alpha_spec.append(combinations_total[i])
    elif i >= 14 and i < 28:
        beta_spec.append(combinations_total[i])
    elif i >= 28 and i < 42:
        theta_spec.append(combinations_total[i])
    elif i >= 42 and i < 43:
        theta_spec.append(combinations_total[i])
        
# print(alpha_spec, beta_spec, theta_spec)
print('Spectrum: ', len(alpha_spec), len(beta_spec), len(theta_spec))

frontal_count, parietal_count = 0, 0
channel_counts = {}
frequency_bands = [alpha_spec, beta_spec, theta_spec]
for frequency_band in frequency_bands:
    for channel in frequency_band:
        # if 'F' in channel:
        #     frontal_count += 1
        # elif 'P' in channel:
        #     parietal_count += 1
        if channel in channel_counts:
            channel_counts[channel] += 1
        else:
            channel_counts[channel] = 1
    
cirkel_diagram(channel_counts, model='Model1')
barplot_features(channel_counts, frequency_bands, node_names_orig, model='Model1')

chosen_frequency_bands1 = [alpha_spec, beta_spec, theta_spec]

# Model 2 - selected spectral features (from baseline) + all 3 frontal-parietal PLV features

In [ ]:
# loading data

directory = '/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'
# directory = '/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'

concatenated_X, concatenated_Y = [], []

for subdir in sorted(os.listdir(directory))[1:]: # index from 1 because Mac has .DS_Store file
    for file in sorted(os.listdir(os.path.join(directory, subdir))):
        data = np.load(os.path.join(directory, subdir, file))
        X_PLV = frontal_parietal_PLV(data['X'][:135]) # 3 PLV features (alpha, beta, theta)
        X_Freq = data['X'][indexes_baseline] # by baseline selected frequency spectrum features
        X = np.concatenate((np.array(X_PLV), np.array(X_Freq)))
        Y = data['Y']
        concatenated_X.append(X)
        concatenated_Y.append(Y)

concatenated_Y = np.concatenate(concatenated_Y)


In [ ]:
# SVM

X = concatenated_X
Y = concatenated_Y

# Create an SVM classifier
classifier = svm.SVC(random_state=42)

# Set the hyperparameters to tune
# parameters = {'C': [0.1, 0.3, 1, 5, 10, 16, 30, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': [0.00001, 0.0005, 0.005, 0.01, 0.05, 0.15, 1, 3, 6]}
parameters = {'C': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.2, 1.5, 1.7, 2, 2.2, 2.5, 2.7, 3, 3.5, 4, 4.5, 5, 6, 7, 8, 9, 10, 12, 15, 17, 20, 22, 25, 27, 30, 35, 40, 45, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': ['auto', 'scale', 0.000001, 0.000002, 0.000003, 0.000004, 0.000005, 0.00001, 0.00002, 0.00003, 0.00004, 0.00005, 0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.001, 0.002, 0.003, 0.004, 0.005, 0.01, 0.02, 0.03, 0.04, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.6, 0.8, 1, 2, 3, 5, 6, 8, 10, 12, 15, 17, 20, 22, 25, 27, 30]}
# Define the scoring metric
scoring = make_scorer(f1_score, average='macro')

# Set the number of folds for cross-validation
num_folds = 10

# Create the Stratified k-fold object
skf = StratifiedKFold(n_splits=num_folds, shuffle=False)

f1_scores, accuracy_scores, precision_scores, recall_scores, cm_scores, best_parameters_list = [],[],[],[],[],[]
k_values = list(range(1, num_folds + 1))

# Perform stratified cross-validation
for k, (train_index, test_index) in enumerate(skf.split(X, Y), start=1):
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    Y_train, Y_test = Y[train_index], Y[test_index]

    # Use grid search to tune the hyperparameters with F1-score as the scoring metric
    grid_search = GridSearchCV(classifier, parameters, scoring=scoring)
    grid_search.fit(X_train, Y_train)

    # Get the best hyperparameters
    best_parameters = grid_search.best_params_

    # Train the classifier using the best hyperparameters
    classifier = svm.SVC(kernel=best_parameters['kernel'], C=best_parameters['C'], gamma=best_parameters['gamma'])
    classifier.fit(X_train, Y_train)

    # Make predictions on the test set
    Y_pred = classifier.predict(X_test)

    # Calculater performance metrics and retrieve best parameter set
    best_parameters_list.append(best_parameters) 
    f1_scores.append(f1_score(Y_test, Y_pred, average='macro'))
    accuracy_scores.append(accuracy_score(Y_test, Y_pred))
    precision_scores.append(precision_score(Y_test, Y_pred, average='macro', zero_division=1))
    recall_scores.append(recall_score(Y_test, Y_pred, average='macro'))
    cm_scores.append(confusion_matrix(Y_test, Y_pred))
    
    # Print the progress
    f1 = f1_score(Y_test, Y_pred, average='macro')
    print(f'Fold {k} - F1-score: {round(f1, 2)} - Best Parameters: {best_parameters}')
    
# print the models performance metrices and its best parameter set
print(' ')
print('Mean F1-score:', f'{round(np.mean(f1_scores), 2)}.', 'Std F1-score:', f'{round(np.std(f1_scores), 2)}.','Max F1-score:', round(np.max(f1_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(f1_scores)]}')
print('Mean Accuracy-score:', f'{round(np.mean(accuracy_scores), 2)}.', 'Std Accuracy-score:', f'{round(np.std(accuracy_scores), 2)}.', 'Max Accuracy-score:', round(np.max(accuracy_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(accuracy_scores)]}')
print('Mean Precision-score:', f'{round(np.mean(precision_scores), 2)}.', 'Std Precision-score:', f'{round(np.std(precision_scores), 2)}.', 'Max Precision-score:', round(np.max(precision_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(precision_scores)]}')
print('Mean Recall-score:', f'{round(np.mean(recall_scores), 2)}.', 'Std Recall-score:', f'{round(np.std(recall_scores), 2)}.', 'Max Recall-score:', round(np.max(recall_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(recall_scores)]}')

elbow_plot(k_values, f1_scores, model='Model2')
cm_plot(cm_scores, f1_scores, model='Model2', total=True)

model2_f1 = f1_scores
model2_param = best_parameters_list[np.argmax(f1_scores)]

In [ ]:
# features selected by the model

# Get the indexes where the array is True
indexes = indexes_baseline

node_names_orig = ['F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'Cz', 'T8', 'P3', 'Pz', 'P4', 'PO7', 'PO8', 'Oz']

# Generate all possible combination pairs of nodes
combinations_total = node_names_orig*3

alpha_spec, beta_spec, theta_spec = [], [], []
for i in indexes:
    if i < 14:
        alpha_spec.append(combinations_total[i])
    elif i >= 14 and i < 28:
        beta_spec.append(combinations_total[i])
    elif i >= 28 and i < 42:
        theta_spec.append(combinations_total[i])

# print(alpha_spec, beta_spec, theta_spec)
print('Spectrum: ', len(alpha_spec), len(beta_spec), len(theta_spec))

frontal_count, parietal_count = 0, 0
channel_counts = {}
frequency_bands = [alpha_spec, beta_spec, theta_spec]
for frequency_band in frequency_bands:
    for channel in frequency_band:
        # if 'F' in channel:
        #     frontal_count += 1
        # elif 'P' in channel:
        #     parietal_count += 1
        if channel in channel_counts:
            channel_counts[channel] += 1
        else:
            channel_counts[channel] = 1
    
cirkel_diagram(channel_counts, model='Model2')
barplot_features(channel_counts, frequency_bands, node_names_orig, model='Model2')

chosen_frequency_bands2 = [alpha_spec, beta_spec, theta_spec]

# Model 3 - RFE on all spectral features + all 3 frontal-parietal PLV features

In [ ]:
# loading data 

directory = '/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'
# directory = '/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'

concatenated_X, concatenated_Y = [], []

for subdir in sorted(os.listdir(directory))[1:]: # index from 1 because Mac has .DS_Store file
    for file in sorted(os.listdir(os.path.join(directory, subdir))):
        data = np.load(os.path.join(directory, subdir, file))
        X_PLV = frontal_parietal_PLV(data['X'][:135]) # 3 PLV features (alpha, beta, theta)
        X_Freq = data['X'][135:] # frequency spectrum features
        X = np.concatenate((np.array(X_PLV), np.array(X_Freq)))
        Y = data['Y']
        concatenated_X.append(X)
        concatenated_Y.append(Y)

concatenated_Y = np.concatenate(concatenated_Y)
    

In [ ]:
# SVM + RFE

X = concatenated_X
Y = concatenated_Y

# Create an SVM classifier
classifier = svm.SVC(random_state=42)

# Set the hyperparameters to tune
# parameters = {'C': [0.1, 0.3, 1, 5, 10, 16, 30, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': [0.00001, 0.0005, 0.005, 0.01, 0.05, 0.15, 1, 3, 6]}
parameters = {'C': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.2, 1.5, 1.7, 2, 2.2, 2.5, 2.7, 3, 3.5, 4, 4.5, 5, 6, 7, 8, 9, 10, 12, 15, 17, 20, 22, 25, 27, 30, 35, 40, 45, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': ['auto', 'scale', 0.000001, 0.000002, 0.000003, 0.000004, 0.000005, 0.00001, 0.00002, 0.00003, 0.00004, 0.00005, 0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.001, 0.002, 0.003, 0.004, 0.005, 0.01, 0.02, 0.03, 0.04, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.6, 0.8, 1, 2, 3, 5, 6, 8, 10, 12, 15, 17, 20, 22, 25, 27, 30]}
# Define the scoring metric
scoring = make_scorer(f1_score, average='macro')

# Set the number of folds for cross-validation
num_folds = 10

# Create the Stratified k-fold object
skf = StratifiedKFold(n_splits=num_folds, shuffle=False)

f1_scores, accuracy_scores, precision_scores, recall_scores, cm_scores, best_parameters_list = [],[],[],[],[],[]
k_values = list(range(1, num_folds + 1))

# Perform stratified cross-validation
for k, (train_index, test_index) in enumerate(skf.split(X, Y), start=1):
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    Y_train, Y_test = Y[train_index], Y[test_index]
    
    # Perform feature selection using RFE with LinearSVC as the estimator
    estimator = svm.LinearSVC(max_iter=30000)  # Use LinearSVC as the estimator for feature selection
    selector = RFE(estimator)
    selector.fit(X_train, Y_train)
    X_train_selected = selector.transform(X_train)
    X_test_selected = selector.transform(X_test)
    
    # Get the selected feature indices
    selected_feature_indices = selector.support_
    selected_feature_ranking = selector.ranking_

    # Use grid search to tune the hyperparameters with F1-score as the scoring metric
    grid_search = GridSearchCV(classifier, parameters, scoring=scoring)
    grid_search.fit(X_train_selected, Y_train)

    # Get the best hyperparameters
    best_parameters = grid_search.best_params_

    # Train the classifier using the best hyperparameters
    classifier = svm.SVC(kernel=best_parameters['kernel'], C=best_parameters['C'], gamma=best_parameters['gamma'])
    classifier.fit(X_train_selected, Y_train)

    # Make predictions on the test set
    Y_pred = classifier.predict(X_test_selected)

    # Calculater performance metrics and retrieve best parameter set
    best_parameters_list.append(best_parameters) 
    f1_scores.append(f1_score(Y_test, Y_pred, average='macro'))
    accuracy_scores.append(accuracy_score(Y_test, Y_pred))
    precision_scores.append(precision_score(Y_test, Y_pred, average='macro', zero_division=1))
    recall_scores.append(recall_score(Y_test, Y_pred, average='macro'))
    cm_scores.append(confusion_matrix(Y_test, Y_pred))
    
    # Print the progress
    f1 = f1_score(Y_test, Y_pred, average='macro')
    print(f'Fold {k} - F1-score: {round(f1, 2)} - Best Parameters: {best_parameters}')
    
# print the models performance metrices and its best parameter set
print(' ')
print('Mean F1-score:', f'{round(np.mean(f1_scores), 2)}.', 'Std F1-score:', f'{round(np.std(f1_scores), 2)}.','Max F1-score:', round(np.max(f1_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(f1_scores)]}')
print('Mean Accuracy-score:', f'{round(np.mean(accuracy_scores), 2)}.', 'Std Accuracy-score:', f'{round(np.std(accuracy_scores), 2)}.', 'Max Accuracy-score:', round(np.max(accuracy_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(accuracy_scores)]}')
print('Mean Precision-score:', f'{round(np.mean(precision_scores), 2)}.', 'Std Precision-score:', f'{round(np.std(precision_scores), 2)}.', 'Max Precision-score:', round(np.max(precision_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(precision_scores)]}')
print('Mean Recall-score:', f'{round(np.mean(recall_scores), 2)}.', 'Std Recall-score:', f'{round(np.std(recall_scores), 2)}.', 'Max Recall-score:', round(np.max(recall_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(recall_scores)]}')

elbow_plot(k_values, f1_scores, model='Model3')
cm_plot(cm_scores, f1_scores, model='Model3', total=True)

model3_f1 = f1_scores
model3_param = best_parameters_list[np.argmax(f1_scores)]

In [ ]:
# features selected by the model

# Get the indexes where the array is True
indexes = np.where(selected_feature_indices)[0]

plv_names = ['alpha-PLV', 'beta-PLV', 'theta-PLV'] 
node_names_orig = ['F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'Cz', 'T8', 'P3', 'Pz', 'P4', 'PO7', 'PO8', 'Oz']

# Generate all possible combination pairs of nodes
combinations_total = plv_names + node_names_orig*3

alpha_plv, beta_plv, theta_plv = [], [], []
alpha_spec, beta_spec, theta_spec = [], [], []
for i in indexes:
    if i < 1:
        alpha_plv.append(combinations_total[i])
    elif i >= 1 and i < 2:
        beta_plv.append(combinations_total[i])
    elif i >= 2 and i < 3:
        theta_plv.append(combinations_total[i])
    elif i >= 3 and i < 17:
        alpha_spec.append(combinations_total[i])
    elif i >= 17 and i < 31:
        beta_spec.append(combinations_total[i])
    elif i >= 31 and i < 45:
        theta_spec.append(combinations_total[i])
        
# print(alpha_comb, beta_comb, theta_comb)
print('PLV: ', len(alpha_plv), len(beta_plv), len(theta_plv))
# print(alpha_spec, beta_spec, theta_spec)
print('Spectrum: ', len(alpha_spec), len(beta_spec), len(theta_spec))
    
frontal_count, parietal_count = 0, 0
channel_counts = {}
frequency_bands = [alpha_spec, beta_spec, theta_spec]
for frequency_band in frequency_bands:
    for channel in frequency_band:
        # if 'F' in channel:
        #     frontal_count += 1
        # elif 'P' in channel:
        #     parietal_count += 1
        if channel in channel_counts:
            channel_counts[channel] += 1
        else:
            channel_counts[channel] = 1
    
cirkel_diagram(channel_counts, model='Model3')
barplot_features(channel_counts, frequency_bands, node_names_orig, model='Model3')

chosen_frequency_bands3 = [alpha_spec, beta_spec, theta_spec]
chosen_plv3 = [alpha_plv, beta_plv, theta_plv]


# Model 4 - RFE on selected spectral features (from baseline) + all 3 frontal-parietal PLV features

In [ ]:
# loading data 

directory = '/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'
# directory = '/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'

concatenated_X, concatenated_Y = [], []

for subdir in sorted(os.listdir(directory))[1:]: # index from 1 because Mac has .DS_Store file
    for file in sorted(os.listdir(os.path.join(directory, subdir))):
        data = np.load(os.path.join(directory, subdir, file))
        X_PLV = frontal_parietal_PLV(data['X'][:135]) # 3 PLV features (alpha, beta, theta)
        X_Freq = data['X'][indexes_baseline] # by baseline selected frequency spectrum features
        X = np.concatenate((np.array(X_PLV), np.array(X_Freq)))
        Y = data['Y']
        concatenated_X.append(X)
        concatenated_Y.append(Y)

concatenated_Y = np.concatenate(concatenated_Y)
    

In [ ]:
# SVM + RFE

X = concatenated_X
Y = concatenated_Y

# Create an SVM classifier
classifier = svm.SVC(random_state=42)

# Set the hyperparameters to tune
# parameters = {'C': [0.1, 0.3, 1, 5, 10, 16, 30, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': [0.00001, 0.0005, 0.005, 0.01, 0.05, 0.15, 1, 3, 6]}
parameters = {'C': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.2, 1.5, 1.7, 2, 2.2, 2.5, 2.7, 3, 3.5, 4, 4.5, 5, 6, 7, 8, 9, 10, 12, 15, 17, 20, 22, 25, 27, 30, 35, 40, 45, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': ['auto', 'scale', 0.000001, 0.000002, 0.000003, 0.000004, 0.000005, 0.00001, 0.00002, 0.00003, 0.00004, 0.00005, 0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.001, 0.002, 0.003, 0.004, 0.005, 0.01, 0.02, 0.03, 0.04, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.6, 0.8, 1, 2, 3, 5, 6, 8, 10, 12, 15, 17, 20, 22, 25, 27, 30]}
# Define the scoring metric
scoring = make_scorer(f1_score, average='macro')

# Set the number of folds for cross-validation
num_folds = 10

# Create the Stratified k-fold object
skf = StratifiedKFold(n_splits=num_folds, shuffle=False)

f1_scores, accuracy_scores, precision_scores, recall_scores, cm_scores, best_parameters_list = [],[],[],[],[],[]
k_values = list(range(1, num_folds + 1))

# Perform stratified cross-validation
for k, (train_index, test_index) in enumerate(skf.split(X, Y), start=1):
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    Y_train, Y_test = Y[train_index], Y[test_index]
    
    # Perform feature selection using RFE with LinearSVC as the estimator
    estimator = svm.LinearSVC(max_iter=30000)  # Use LinearSVC as the estimator for feature selection
    selector = RFE(estimator)
    selector.fit(X_train, Y_train)
    X_train_selected = selector.transform(X_train)
    X_test_selected = selector.transform(X_test)
    
    # Get the selected feature indices
    selected_feature_indices = selector.support_
    selected_feature_ranking = selector.ranking_

    # Use grid search to tune the hyperparameters with F1-score as the scoring metric
    grid_search = GridSearchCV(classifier, parameters, scoring=scoring)
    grid_search.fit(X_train_selected, Y_train)

    # Get the best hyperparameters
    best_parameters = grid_search.best_params_

    # Train the classifier using the best hyperparameters
    classifier = svm.SVC(kernel=best_parameters['kernel'], C=best_parameters['C'], gamma=best_parameters['gamma'])
    classifier.fit(X_train_selected, Y_train)

    # Make predictions on the test set
    Y_pred = classifier.predict(X_test_selected)

    # Calculater performance metrics and retrieve best parameter set
    best_parameters_list.append(best_parameters) 
    f1_scores.append(f1_score(Y_test, Y_pred, average='macro'))
    accuracy_scores.append(accuracy_score(Y_test, Y_pred))
    precision_scores.append(precision_score(Y_test, Y_pred, average='macro', zero_division=1))
    recall_scores.append(recall_score(Y_test, Y_pred, average='macro'))
    cm_scores.append(confusion_matrix(Y_test, Y_pred))
    
    # Print the progress
    f1 = f1_score(Y_test, Y_pred, average='macro')
    print(f'Fold {k} - F1-score: {round(f1, 2)} - Best Parameters: {best_parameters}')
    
# print the models performance metrices and its best parameter set
print(' ')
print('Mean F1-score:', f'{round(np.mean(f1_scores), 2)}.', 'Std F1-score:', f'{round(np.std(f1_scores), 2)}.','Max F1-score:', round(np.max(f1_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(f1_scores)]}')
print('Mean Accuracy-score:', f'{round(np.mean(accuracy_scores), 2)}.', 'Std Accuracy-score:', f'{round(np.std(accuracy_scores), 2)}.', 'Max Accuracy-score:', round(np.max(accuracy_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(accuracy_scores)]}')
print('Mean Precision-score:', f'{round(np.mean(precision_scores), 2)}.', 'Std Precision-score:', f'{round(np.std(precision_scores), 2)}.', 'Max Precision-score:', round(np.max(precision_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(precision_scores)]}')
print('Mean Recall-score:', f'{round(np.mean(recall_scores), 2)}.', 'Std Recall-score:', f'{round(np.std(recall_scores), 2)}.', 'Max Recall-score:', round(np.max(recall_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(recall_scores)]}')

elbow_plot(k_values, f1_scores, model='Model4')
cm_plot(cm_scores, f1_scores, model='Model4', total=True)

model4_f1 = f1_scores
model4_param = best_parameters_list[np.argmax(f1_scores)]

In [ ]:
# features selected by the model
indexes = indexes_baseline
node_names_orig = ['F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'Cz', 'T8', 'P3', 'Pz', 'P4', 'PO7', 'PO8', 'Oz']
combinations_total = node_names_orig*3
alpha_spec, beta_spec, theta_spec = [], [], []
for i in indexes:
    if i < 14:
        alpha_spec.append(combinations_total[i])
    elif i >= 14 and i < 28:
        beta_spec.append(combinations_total[i])
    elif i >= 28 and i < 42:
        theta_spec.append(combinations_total[i])

# Get the indexes where the array is True
indexes = np.where(selected_feature_indices)[0]
plv_names = ['alpha-PLV', 'beta-PLV', 'theta-PLV'] 
# # Generate all possible combination pairs of nodes
combinations_total = plv_names + alpha_spec + beta_spec + theta_spec
alpha_plv, beta_plv, theta_plv = [], [], []
alpha_spec1, beta_spec1, theta_spec1 = [], [], []
for i in indexes:
    if i < 1:
        alpha_plv.append(combinations_total[i])
    elif i >= 1 and i < 2:
        beta_plv.append(combinations_total[i])
    elif i >= 2 and i < 3:
        theta_plv.append(combinations_total[i])
    elif i >= 3 and i < 3+len(alpha_spec):
        alpha_spec1.append(combinations_total[i])
    elif i >= 3+len(alpha_spec) and i < 3+len(alpha_spec)+len(beta_spec):
        beta_spec1.append(combinations_total[i])
    elif i >= 3+len(alpha_spec)+len(beta_spec) and i < 3+len(alpha_spec)+len(beta_spec)+len(theta_spec):
        theta_spec1.append(combinations_total[i])
        
# print(alpha_comb, beta_comb, theta_comb)
print('PLV: ', len(alpha_plv), len(beta_plv), len(theta_plv))
# print(alpha_spec, beta_spec, theta_spec)
print('Spectrum: ', len(alpha_spec1), len(beta_spec1), len(theta_spec1))
    
frontal_count, parietal_count = 0, 0
channel_counts = {}
frequency_bands = [alpha_spec1, beta_spec1, theta_spec1]
for frequency_band in frequency_bands:
    for channel in frequency_band:
        # if 'F' in channel:
        #     frontal_count += 1
        # elif 'P' in channel:
        #     parietal_count += 1
        if channel in channel_counts:
            channel_counts[channel] += 1
        else:
            channel_counts[channel] = 1
    
cirkel_diagram(channel_counts, model='Model4')
barplot_features(channel_counts, frequency_bands, node_names_orig, model='Model4')

chosen_frequency_bands4 = [alpha_spec1, beta_spec1, theta_spec1]
chosen_plv4 = [alpha_plv, beta_plv, theta_plv]


# Model 5 - selected spectral features (from baseline) + RFE on all 3 PLV frontal-parietal features

In [ ]:
# loading data 

directory = '/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'
# directory = '/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'

concatenated_X, concatenated_Y = [], []

for subdir in sorted(os.listdir(directory))[1:]: # index from 1 because Mac has .DS_Store file
    for file in sorted(os.listdir(os.path.join(directory, subdir))):
        data = np.load(os.path.join(directory, subdir, file))
        X_PLV = frontal_parietal_PLV(data['X'][:135]) # 3 PLV features (alpha, beta, theta)
        X_Freq = data['X'][indexes_baseline] # by baseline selected frequency spectrum features
        X = np.concatenate((X_PLV, X_Freq))
        Y = data['Y']
        concatenated_X.append(X)
        concatenated_Y.append(Y)

concatenated_Y = np.concatenate(concatenated_Y)
    

In [ ]:
# SVM + RFE

X = concatenated_X
Y = concatenated_Y

# Create an SVM classifier
classifier = svm.SVC(random_state=42)

# Set the hyperparameters to tune
# parameters = {'C': [0.1, 0.3, 1, 5, 10, 16, 30, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': [0.00001, 0.0005, 0.005, 0.01, 0.05, 0.15, 1, 3, 6]}
parameters = {'C': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.2, 1.5, 1.7, 2, 2.2, 2.5, 2.7, 3, 3.5, 4, 4.5, 5, 6, 7, 8, 9, 10, 12, 15, 17, 20, 22, 25, 27, 30, 35, 40, 45, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': ['auto', 'scale', 0.000001, 0.000002, 0.000003, 0.000004, 0.000005, 0.00001, 0.00002, 0.00003, 0.00004, 0.00005, 0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.001, 0.002, 0.003, 0.004, 0.005, 0.01, 0.02, 0.03, 0.04, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.6, 0.8, 1, 2, 3, 5, 6, 8, 10, 12, 15, 17, 20, 22, 25, 27, 30]}

# Define the scoring metric
scoring = make_scorer(f1_score, average='macro')

# Set the number of folds for cross-validation
num_folds = 10

# Create the Stratified k-fold object
skf = StratifiedKFold(n_splits=num_folds, shuffle=False)

f1_scores, accuracy_scores, precision_scores, recall_scores, cm_scores, best_parameters_list = [],[],[],[],[],[]
k_values = list(range(1, num_folds + 1))

# Perform stratified cross-validation
for k, (train_index, test_index) in enumerate(skf.split(X, Y), start=1):
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    Y_train, Y_test = Y[train_index], Y[test_index]
    
    # Perform feature selection using RFE with LinearSVC as the estimator
    estimator = svm.LinearSVC(max_iter=30000)  # Use LinearSVC as the estimator for feature selection
    selector = RFE(estimator)
    # Select the PLV features in X_train and X_test
    X_train_plv = X_train[:, :3]
    X_test_plv = X_test[:, :3]

    # Perform feature selection on the selected features
    selector.fit(X_train_plv, Y_train)

    # Get the selected feature indices
    selected_feature_indices = selector.support_
    selected_feature_ranking = selector.ranking_

    # Apply feature selection on X_train and X_test
    X_train_plv_selected = X_train_plv[:, selected_feature_indices]
    X_test_plv_selected = X_test_plv[:, selected_feature_indices]

    # Concatenate the remaining features with the selected features
    X_train_selected = np.concatenate((X_train_plv_selected, X_train[:, 30:]), axis=1)
    X_test_selected = np.concatenate((X_test_plv_selected, X_test[:, 30:]), axis=1)

    # Use grid search to tune the hyperparameters with F1-score as the scoring metric
    grid_search = GridSearchCV(classifier, parameters, scoring=scoring)
    grid_search.fit(X_train_selected, Y_train)

    # Get the best hyperparameters
    best_parameters = grid_search.best_params_

    # Train the classifier using the best hyperparameters
    classifier = svm.SVC(kernel=best_parameters['kernel'], C=best_parameters['C'], gamma=best_parameters['gamma'])
    classifier.fit(X_train_selected, Y_train)

    # Make predictions on the test set
    Y_pred = classifier.predict(X_test_selected)

    # Calculater performance metrics and retrieve best parameter set
    best_parameters_list.append(best_parameters) 
    f1_scores.append(f1_score(Y_test, Y_pred, average='macro'))
    accuracy_scores.append(accuracy_score(Y_test, Y_pred))
    precision_scores.append(precision_score(Y_test, Y_pred, average='macro', zero_division=1))
    recall_scores.append(recall_score(Y_test, Y_pred, average='macro'))
    cm_scores.append(confusion_matrix(Y_test, Y_pred))
    
    # Print the progress
    f1 = f1_score(Y_test, Y_pred, average='macro')
    print(f'Fold {k} - F1-score: {round(f1, 2)} - Best Parameters: {best_parameters}')
    
# print the models performance metrices and its best parameter set
print(' ')
print('Mean F1-score:', f'{round(np.mean(f1_scores), 2)}.', 'Std F1-score:', f'{round(np.std(f1_scores), 2)}.','Max F1-score:', round(np.max(f1_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(f1_scores)]}')
print(' ')
print('Mean Accuracy-score:', f'{round(np.mean(accuracy_scores), 2)}.', 'Std Accuracy-score:', f'{round(np.std(accuracy_scores), 2)}.', 'Max Accuracy-score:', round(np.max(accuracy_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(accuracy_scores)]}')
print(' ')
print('Mean Precision-score:', f'{round(np.mean(precision_scores), 2)}.', 'Std Precision-score:', f'{round(np.std(precision_scores), 2)}.', 'Max Precision-score:', round(np.max(precision_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(precision_scores)]}')
print(' ')
print('Mean Recall-score:', f'{round(np.mean(recall_scores), 2)}.', 'Std Recall-score:', f'{round(np.std(recall_scores), 2)}.', 'Max Recall-score:', round(np.max(recall_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(recall_scores)]}')
print(' ')

elbow_plot(k_values, f1_scores, model='Model5')
cm_plot(cm_scores, f1_scores, model='Model5', total=True)

model5_f1 = f1_scores
model5_param = best_parameters_list[np.argmax(f1_scores)]

In [ ]:
# features selected by the model
selected_node_names_orig = []

plv_names = ['alpha-PLV', 'beta-PLV', 'theta-PLV'] 
indexes_plv = np.where(selected_feature_indices)[0]
alpha_plv, beta_plv, theta_plv = [], [], []
for i in indexes_plv:
    if i == 0:
        alpha_plv.append(plv_names[i])
    elif i == 1:
        beta_plv.append(plv_names[i])
    elif i == 2:
        theta_plv.append(plv_names[i])
        
chosen_plv5 = [alpha_plv, beta_plv, theta_plv]

# Get the indexes where the array is True
indexes = indexes_baseline
node_names_orig = ['F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'Cz', 'T8', 'P3', 'Pz', 'P4', 'PO7', 'PO8', 'Oz']

# Generate all possible combination pairs of nodes
combinations_total = node_names_orig*3

alpha_spec, beta_spec, theta_spec = [], [], []
for i in indexes:
    if i < 14:
        alpha_spec.append(combinations_total[i])
    elif i >= 14 and i < 28:
        beta_spec.append(combinations_total[i])
    elif i >= 28 and i < 42:
        theta_spec.append(combinations_total[i])
        
# print(alpha_comb, beta_comb, theta_comb)
print('PLV: ', len(alpha_plv), len(beta_plv), len(theta_plv))
# print(alpha_spec, beta_spec, theta_spec)
print('Spectrum: ', len(alpha_spec), len(beta_spec), len(theta_spec))
    
frontal_count, parietal_count = 0, 0
channel_counts = {}
frequency_bands = [alpha_spec, beta_spec, theta_spec]
for frequency_band in frequency_bands:
    for channel in frequency_band:
        # if 'F' in channel:
        #     frontal_count += 1
        # elif 'P' in channel:
        #     parietal_count += 1
        if channel in channel_counts:
            channel_counts[channel] += 1
        else:
            channel_counts[channel] = 1
    
# cirkel_diagram(channel_counts, model='Model5')
# barplot_features(channel_counts, frequency_bands, node_names_orig, model='Model5')

chosen_frequency_bands5 = [alpha_spec, beta_spec, theta_spec]

# Model 6 - all spectral features + all 30 frontal-parietal PLV features

In [ ]:
# loading data 

directory = '/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'
# directory = '/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'

concatenated_X, concatenated_Y = [], []

for subdir in sorted(os.listdir(directory))[1:]: # index from 1 because Mac has .DS_Store file
    for file in sorted(os.listdir(os.path.join(directory, subdir))):
        data = np.load(os.path.join(directory, subdir, file))
        X_PLV = frontal_parietal_PLVs(data['X'][:135]) # 30 PLV features (10 alpha, 10 beta, 10 theta)
        X_Freq = data['X'][135:] # frequency spectrum features
        X = np.concatenate((X_PLV, X_Freq))
        Y = data['Y']
        concatenated_X.append(X)
        concatenated_Y.append(Y)

concatenated_Y = np.concatenate(concatenated_Y)


In [ ]:
# SVM

X = concatenated_X
Y = concatenated_Y

# Create an SVM classifier
classifier = svm.SVC(random_state=42)

# Set the hyperparameters to tune
# parameters = {'C': [0.1, 0.3, 1, 5, 10, 16, 30, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': [0.00001, 0.0005, 0.005, 0.01, 0.05, 0.15, 1, 3, 6]}
parameters = {'C': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.2, 1.5, 1.7, 2, 2.2, 2.5, 2.7, 3, 3.5, 4, 4.5, 5, 6, 7, 8, 9, 10, 12, 15, 17, 20, 22, 25, 27, 30, 35, 40, 45, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': ['auto', 'scale', 0.000001, 0.000002, 0.000003, 0.000004, 0.000005, 0.00001, 0.00002, 0.00003, 0.00004, 0.00005, 0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.001, 0.002, 0.003, 0.004, 0.005, 0.01, 0.02, 0.03, 0.04, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.6, 0.8, 1, 2, 3, 5, 6, 8, 10, 12, 15, 17, 20, 22, 25, 27, 30]}

# Define the scoring metric
scoring = make_scorer(f1_score, average='macro')

# Set the number of folds for cross-validation
num_folds = 10

# Create the Stratified k-fold object
skf = StratifiedKFold(n_splits=num_folds, shuffle=False)

f1_scores, accuracy_scores, precision_scores, recall_scores, cm_scores, best_parameters_list = [],[],[],[],[],[]
k_values = list(range(1, num_folds + 1))

# Perform stratified cross-validation
for k, (train_index, test_index) in enumerate(skf.split(X, Y), start=1):
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    Y_train, Y_test = Y[train_index], Y[test_index]

    # Use grid search to tune the hyperparameters with F1-score as the scoring metric
    grid_search = GridSearchCV(classifier, parameters, scoring=scoring)
    grid_search.fit(X_train, Y_train)

    # Get the best hyperparameters
    best_parameters = grid_search.best_params_

    # Train the classifier using the best hyperparameters
    classifier = svm.SVC(kernel=best_parameters['kernel'], C=best_parameters['C'], gamma=best_parameters['gamma'])
    classifier.fit(X_train, Y_train)

    # Make predictions on the test set
    Y_pred = classifier.predict(X_test)

    # Calculater performance metrics and retrieve best parameter set
    best_parameters_list.append(best_parameters) 
    f1_scores.append(f1_score(Y_test, Y_pred, average='macro'))
    accuracy_scores.append(accuracy_score(Y_test, Y_pred))
    precision_scores.append(precision_score(Y_test, Y_pred, average='macro', zero_division=1))
    recall_scores.append(recall_score(Y_test, Y_pred, average='macro'))
    cm_scores.append(confusion_matrix(Y_test, Y_pred))
    
    # Print the progress
    f1 = f1_score(Y_test, Y_pred, average='macro')
    print(f'Fold {k} - F1-score: {round(f1, 2)} - Best Parameters: {best_parameters}')
    
# print the models performance metrices and its best parameter set
print(' ')
print('Mean F1-score:', f'{round(np.mean(f1_scores), 2)}.', 'Std F1-score:', f'{round(np.std(f1_scores), 2)}.','Max F1-score:', round(np.max(f1_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(f1_scores)]}')
print(' ')
print('Mean Accuracy-score:', f'{round(np.mean(accuracy_scores), 2)}.', 'Std Accuracy-score:', f'{round(np.std(accuracy_scores), 2)}.', 'Max Accuracy-score:', round(np.max(accuracy_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(accuracy_scores)]}')
print(' ')
print('Mean Precision-score:', f'{round(np.mean(precision_scores), 2)}.', 'Std Precision-score:', f'{round(np.std(precision_scores), 2)}.', 'Max Precision-score:', round(np.max(precision_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(precision_scores)]}')
print(' ')
print('Mean Recall-score:', f'{round(np.mean(recall_scores), 2)}.', 'Std Recall-score:', f'{round(np.std(recall_scores), 2)}.', 'Max Recall-score:', round(np.max(recall_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(recall_scores)]}')
print(' ')

elbow_plot(k_values, f1_scores, model='Model6')
cm_plot(cm_scores, f1_scores, model='Model6', total=True)

model6_f1 = f1_scores
model6_param = best_parameters_list[np.argmax(f1_scores)]

In [ ]:
# features selected by the model

plv_names = ['alpha-F7','alpha-F3','alpha-Fz','alpha-F4','alpha-F8',
             'alpha-P3','alpha-Pz','alpha-P4','alpha-PO7','alpha-PO8',
             'beta-F7','beta-F3','beta-Fz','beta-F4','beta-F8',
             'beta-P3','beta-Pz','beta-P4','beta-PO7','beta-PO8', 
             'theta-F7','theta-F3','theta-Fz','theta-F4','theta-F8',
             'theta-P3','theta-Pz','theta-P4','theta-PO7','theta-PO8'] 
node_names_orig = ['F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'Cz', 'T8', 'P3', 'Pz', 'P4', 'PO7', 'PO8', 'Oz']

# Generate all possible combination pairs of nodes
combinations_total = plv_names + node_names_orig*3

alpha_spec, beta_spec, theta_spec = [], [], []
alpha_plv, beta_plv, theta_plv = [], [], []
for i in range(len(combinations_total)):
    if i < 10:
        alpha_plv.append(combinations_total[i])
    elif i >= 10 and i < 20:
        beta_plv.append(combinations_total[i])
    elif i >= 20 and i < 30:
        theta_plv.append(combinations_total[i])
    elif i >= 30 and i < 44:
        alpha_spec.append(combinations_total[i])
    elif i >= 44 and i < 58:
        beta_spec.append(combinations_total[i])
    elif i >= 58 and i < 72:
        theta_spec.append(combinations_total[i])
        
# print(alpha_plv, beta_plv, theta_plv)
print('PLV: ', len(alpha_plv), len(beta_plv), len(theta_plv))
# print(alpha_spec, beta_spec, theta_spec)
print('Spectrum: ', len(alpha_spec), len(beta_spec), len(theta_spec))
    
frontal_count, parietal_count = 0, 0
channel_counts = {}
frequency_bands = [alpha_spec, beta_spec, theta_spec]
for frequency_band in frequency_bands:
    for channel in frequency_band:
        # if 'F' in channel:
        #     frontal_count += 1
        # elif 'P' in channel:
        #     parietal_count += 1
        if channel in channel_counts:
            channel_counts[channel] += 1
        else:
            channel_counts[channel] = 1
    
# cirkel_diagram(channel_counts, model='Model6')
# barplot_features(channel_counts, frequency_bands, node_names_orig, model='Model6')

chosen_frequency_bands6 = [alpha_spec, beta_spec, theta_spec]

frontal_count, parietal_count = 0, 0
channel_counts = {}
frequency_bands = [alpha_plv, beta_plv, theta_plv]
for frequency_band in frequency_bands:
    for channel in frequency_band:
        # if 'F' in channel:
        #     frontal_count += 1
        # elif 'P' in channel:
        #     parietal_count += 1
        if channel in channel_counts:
            channel_counts[channel] += 1
        else:
            channel_counts[channel] = 1
            
# barplot_PLVfeatures(channel_counts, frequency_bands, plv_names, model='Model6')

chosen_plv6 = [alpha_plv, beta_plv, theta_plv]


# Model 7 - selected spectral features (from baseline) + all 30 frontal-parietal PLV features

In [ ]:
# loading data
directory = '/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'
# directory = '/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'

concatenated_X, concatenated_Y = [], []

for subdir in sorted(os.listdir(directory))[1:]: # index from 1 because Mac has .DS_Store file
    for file in sorted(os.listdir(os.path.join(directory, subdir))):
        data = np.load(os.path.join(directory, subdir, file))
        X_PLV = frontal_parietal_PLVs(data['X'][:135]) # 30 PLV features (10 alpha, 10 beta, 10 theta)
        X_Freq = data['X'][indexes_baseline] # by baseline selected frequency spectrum features
        X = np.concatenate((X_PLV, X_Freq))
        Y = data['Y']
        concatenated_X.append(X)
        concatenated_Y.append(Y)

concatenated_Y = np.concatenate(concatenated_Y)
    

In [ ]:
# SVM

X = concatenated_X
Y = concatenated_Y

# Create an SVM classifier
classifier = svm.SVC(random_state=42)

# Set the hyperparameters to tune
# parameters = {'C': [0.1, 0.3, 1, 5, 10, 16, 30, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': [0.00001, 0.0005, 0.005, 0.01, 0.05, 0.15, 1, 3, 6]}
parameters = {'C': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.2, 1.5, 1.7, 2, 2.2, 2.5, 2.7, 3, 3.5, 4, 4.5, 5, 6, 7, 8, 9, 10, 12, 15, 17, 20, 22, 25, 27, 30, 35, 40, 45, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': ['auto', 'scale', 0.000001, 0.000002, 0.000003, 0.000004, 0.000005, 0.00001, 0.00002, 0.00003, 0.00004, 0.00005, 0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.001, 0.002, 0.003, 0.004, 0.005, 0.01, 0.02, 0.03, 0.04, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.6, 0.8, 1, 2, 3, 5, 6, 8, 10, 12, 15, 17, 20, 22, 25, 27, 30]}

# Define the scoring metric
scoring = make_scorer(f1_score, average='macro')

# Set the number of folds for cross-validation
num_folds = 10

# Create the Stratified k-fold object
skf = StratifiedKFold(n_splits=num_folds, shuffle=False)

f1_scores, accuracy_scores, precision_scores, recall_scores, cm_scores, best_parameters_list = [],[],[],[],[],[]
k_values = list(range(1, num_folds + 1))

# Perform stratified cross-validation
for k, (train_index, test_index) in enumerate(skf.split(X, Y), start=1):
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    Y_train, Y_test = Y[train_index], Y[test_index]

    # Use grid search to tune the hyperparameters with F1-score as the scoring metric
    grid_search = GridSearchCV(classifier, parameters, scoring=scoring)
    grid_search.fit(X_train, Y_train)

    # Get the best hyperparameters
    best_parameters = grid_search.best_params_

    # Train the classifier using the best hyperparameters
    classifier = svm.SVC(kernel=best_parameters['kernel'], C=best_parameters['C'], gamma=best_parameters['gamma'])
    classifier.fit(X_train, Y_train)

    # Make predictions on the test set
    Y_pred = classifier.predict(X_test)

    # Calculater performance metrics and retrieve best parameter set
    best_parameters_list.append(best_parameters) 
    f1_scores.append(f1_score(Y_test, Y_pred, average='macro'))
    accuracy_scores.append(accuracy_score(Y_test, Y_pred))
    precision_scores.append(precision_score(Y_test, Y_pred, average='macro', zero_division=1))
    recall_scores.append(recall_score(Y_test, Y_pred, average='macro'))
    cm_scores.append(confusion_matrix(Y_test, Y_pred))
    
    # Print the progress
    f1 = f1_score(Y_test, Y_pred, average='macro')
    print(f'Fold {k} - F1-score: {round(f1, 2)} - Best Parameters: {best_parameters}')
    
# print the models performance metrices and its best parameter set
print(' ')
print('Mean F1-score:', f'{round(np.mean(f1_scores), 2)}.', 'Std F1-score:', f'{round(np.std(f1_scores), 2)}.','Max F1-score:', round(np.max(f1_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(f1_scores)]}')
print(' ')
print('Mean Accuracy-score:', f'{round(np.mean(accuracy_scores), 2)}.', 'Std Accuracy-score:', f'{round(np.std(accuracy_scores), 2)}.', 'Max Accuracy-score:', round(np.max(accuracy_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(accuracy_scores)]}')
print(' ')
print('Mean Precision-score:', f'{round(np.mean(precision_scores), 2)}.', 'Std Precision-score:', f'{round(np.std(precision_scores), 2)}.', 'Max Precision-score:', round(np.max(precision_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(precision_scores)]}')
print(' ')
print('Mean Recall-score:', f'{round(np.mean(recall_scores), 2)}.', 'Std Recall-score:', f'{round(np.std(recall_scores), 2)}.', 'Max Recall-score:', round(np.max(recall_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(recall_scores)]}')
print(' ')

elbow_plot(k_values, f1_scores, model='Model7')
cm_plot(cm_scores, f1_scores, model='Model7', total=True)

model7_f1 = f1_scores
model7_param = best_parameters_list[np.argmax(f1_scores)]

In [ ]:
# features selected by the model

# Get the indexes where the array is True
indexes = indexes_baseline

plv_names = ['alpha-F7','alpha-F3','alpha-Fz','alpha-F4','alpha-F8',
             'alpha-P3','alpha-Pz','alpha-P4','alpha-PO7','alpha-PO8',
             'beta-F7','beta-F3','beta-Fz','beta-F4','beta-F8',
             'beta-P3','beta-Pz','beta-P4','beta-PO7','beta-PO8', 
             'theta-F7','theta-F3','theta-Fz','theta-F4','theta-F8',
             'theta-P3','theta-Pz','theta-P4','theta-PO7','theta-PO8'] 
node_names_orig = ['F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'Cz', 'T8', 'P3', 'Pz', 'P4', 'PO7', 'PO8', 'Oz']
tot = node_names_orig*3
selected_spectrum_feat = [[],[],[]]
for i in indexes_baseline:
    if i >= 0 and i < 14:
        selected_spectrum_feat[0].append(tot[i])
    elif i >= 14 and i < 28:
        selected_spectrum_feat[1].append(tot[i])
    elif i >= 28 and i < 42:
        selected_spectrum_feat[2].append(tot[i])

tot_nodes = [plv_names] + selected_spectrum_feat
selected_all_feat = [[], [], [], [], [], []]
for i in range(len(plv_names)+len(indexes_baseline)):
    if i < 10:
        selected_all_feat[0].append(tot_nodes[0][i])
    elif i >= 10 and i < 20:
        selected_all_feat[1].append(tot_nodes[0][i])
    elif i >= 20 and i < 30:
        selected_all_feat[2].append(tot_nodes[0][i])
    elif i >= 30 and i < (30 + len(selected_spectrum_feat[0])):
        selected_all_feat[3].append(tot_nodes[1][i-30])
    elif i >= 30 + len(selected_spectrum_feat[0]) and i < 30 + len(selected_spectrum_feat[0]) + len(selected_spectrum_feat[1]):
        selected_all_feat[4].append(tot_nodes[2][i-30-len(selected_spectrum_feat[0])])
    elif i >= 30 + len(selected_spectrum_feat[0]) + len(selected_spectrum_feat[1]) and i < 30 + len(selected_spectrum_feat[0]) + len(selected_spectrum_feat[1]) + len(selected_spectrum_feat[2]):
        selected_all_feat[5].append(tot_nodes[3][i-30-len(selected_spectrum_feat[0])-len(selected_spectrum_feat[1])])
        
# print(alpha_comb, beta_comb, theta_comb)
print('PLV: ', len(selected_all_feat[0]), len(selected_all_feat[1]), len(selected_all_feat[2]))
# print(alpha_spec, beta_spec, theta_spec)
print('Spectrum: ', len(selected_all_feat[3]), len(selected_all_feat[4]), len(selected_all_feat[5]))
    
frontal_count, parietal_count = 0, 0
channel_counts = {}
frequency_bands = [selected_all_feat[3], selected_all_feat[4], selected_all_feat[5]]
for frequency_band in frequency_bands:
    for channel in frequency_band:
        # if 'F' in channel:
        #     frontal_count += 1
        # elif 'P' in channel:
        #     parietal_count += 1
        if channel in channel_counts:
            channel_counts[channel] += 1
        else:
            channel_counts[channel] = 1
    
# cirkel_diagram(channel_counts, model='Model7')
# barplot_features(channel_counts, frequency_bands, node_names_orig, model='Model7')

chosen_frequency_bands7 = [alpha_spec, beta_spec, theta_spec]

frontal_count, parietal_count = 0, 0
channel_counts = {}
frequency_bands = [selected_all_feat[0], selected_all_feat[1], selected_all_feat[2]]
for frequency_band in frequency_bands:
    for channel in frequency_band:
        # if 'F' in channel:
        #     frontal_count += 1
        # elif 'P' in channel:
        #     parietal_count += 1
        if channel in channel_counts:
            channel_counts[channel] += 1
        else:
            channel_counts[channel] = 1
    
# barplot_PLVfeatures(channel_counts, frequency_bands, plv_names, model='Model7')

chosen_plv7 = [selected_all_feat[0], selected_all_feat[1], selected_all_feat[2]]


# Model 8 - RFE on all spectral features + all 30 frontal-parietal PLV features

In [ ]:
# loading data 

directory = '/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'
# directory = '/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'

concatenated_X, concatenated_Y = [], []

for subdir in sorted(os.listdir(directory))[1:]: # index from 1 because Mac has .DS_Store file
    for file in sorted(os.listdir(os.path.join(directory, subdir))):
        data = np.load(os.path.join(directory, subdir, file))
        X_PLV = frontal_parietal_PLVs(data['X'][:135]) # 30 PLV features (10 alpha, 10 beta, 10 theta)
        X_Freq = data['X'][135:] # frequency spectrum features
        X = np.concatenate((X_PLV, X_Freq))
        Y = data['Y']
        concatenated_X.append(X)
        concatenated_Y.append(Y)

concatenated_Y = np.concatenate(concatenated_Y)


In [ ]:
# SVM + RFE

X = concatenated_X
Y = concatenated_Y

# Create an SVM classifier
classifier = svm.SVC(random_state=42)

# Set the hyperparameters to tune
# parameters = {'C': [0.1, 0.3, 1, 5, 10, 16, 30, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': [0.00001, 0.0005, 0.005, 0.01, 0.05, 0.15, 1, 3, 6]}
parameters = {'C': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.2, 1.5, 1.7, 2, 2.2, 2.5, 2.7, 3, 3.5, 4, 4.5, 5, 6, 7, 8, 9, 10, 12, 15, 17, 20, 22, 25, 27, 30, 35, 40, 45, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': ['auto', 'scale', 0.000001, 0.000002, 0.000003, 0.000004, 0.000005, 0.00001, 0.00002, 0.00003, 0.00004, 0.00005, 0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.001, 0.002, 0.003, 0.004, 0.005, 0.01, 0.02, 0.03, 0.04, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.6, 0.8, 1, 2, 3, 5, 6, 8, 10, 12, 15, 17, 20, 22, 25, 27, 30]}

# Define the scoring metric
scoring = make_scorer(f1_score, average='macro')

# Set the number of folds for cross-validation
num_folds = 10

# Create the Stratified k-fold object
skf = StratifiedKFold(n_splits=num_folds, shuffle=False)

f1_scores, accuracy_scores, precision_scores, recall_scores, cm_scores, best_parameters_list = [],[],[],[],[],[]
k_values = list(range(1, num_folds + 1))

# Perform stratified cross-validation
for k, (train_index, test_index) in enumerate(skf.split(X, Y), start=1):
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    Y_train, Y_test = Y[train_index], Y[test_index]
    
    # Perform feature selection using RFE with LinearSVC as the estimator
    estimator = svm.LinearSVC(max_iter=30000)  # Use LinearSVC as the estimator for feature selection
    selector = RFE(estimator)
    selector.fit(X_train, Y_train)
    X_train_selected = selector.transform(X_train)
    X_test_selected = selector.transform(X_test)
    
    # Get the selected feature indices
    selected_feature_indices = selector.support_
    selected_feature_ranking = selector.ranking_

    # Use grid search to tune the hyperparameters with F1-score as the scoring metric
    grid_search = GridSearchCV(classifier, parameters, scoring=scoring)
    grid_search.fit(X_train_selected, Y_train)

    # Get the best hyperparameters
    best_parameters = grid_search.best_params_

    # Train the classifier using the best hyperparameters
    classifier = svm.SVC(kernel=best_parameters['kernel'], C=best_parameters['C'], gamma=best_parameters['gamma'])
    classifier.fit(X_train_selected, Y_train)

    # Make predictions on the test set
    Y_pred = classifier.predict(X_test_selected)

    # Calculater performance metrics and retrieve best parameter set
    best_parameters_list.append(best_parameters) 
    f1_scores.append(f1_score(Y_test, Y_pred, average='macro'))
    accuracy_scores.append(accuracy_score(Y_test, Y_pred))
    precision_scores.append(precision_score(Y_test, Y_pred, average='macro', zero_division=1))
    recall_scores.append(recall_score(Y_test, Y_pred, average='macro'))
    cm_scores.append(confusion_matrix(Y_test, Y_pred))
    
    # Print the progress
    f1 = f1_score(Y_test, Y_pred, average='macro')
    print(f'Fold {k} - F1-score: {round(f1, 2)} - Best Parameters: {best_parameters}')
    
# print the models performance metrices and its best parameter set
print(' ')
print('Mean F1-score:', f'{round(np.mean(f1_scores), 2)}.', 'Std F1-score:', f'{round(np.std(f1_scores), 2)}.','Max F1-score:', round(np.max(f1_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(f1_scores)]}')
print(' ')
print('Mean Accuracy-score:', f'{round(np.mean(accuracy_scores), 2)}.', 'Std Accuracy-score:', f'{round(np.std(accuracy_scores), 2)}.', 'Max Accuracy-score:', round(np.max(accuracy_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(accuracy_scores)]}')
print(' ')
print('Mean Precision-score:', f'{round(np.mean(precision_scores), 2)}.', 'Std Precision-score:', f'{round(np.std(precision_scores), 2)}.', 'Max Precision-score:', round(np.max(precision_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(precision_scores)]}')
print(' ')
print('Mean Recall-score:', f'{round(np.mean(recall_scores), 2)}.', 'Std Recall-score:', f'{round(np.std(recall_scores), 2)}.', 'Max Recall-score:', round(np.max(recall_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(recall_scores)]}')
print(' ')

elbow_plot(k_values, f1_scores, model='Model8')
cm_plot(cm_scores, f1_scores, model='Model8', total=True)

model8_f1 = f1_scores
model8_param = best_parameters_list[np.argmax(f1_scores)]

In [ ]:
# features selected by the model

# Get the indexes where the array is True
indexes = np.where(selected_feature_indices)[0]

plv_names = ['alpha-F7','alpha-F3','alpha-Fz','alpha-F4','alpha-F8',
             'alpha-P3','alpha-Pz','alpha-P4','alpha-PO7','alpha-PO8',
             'beta-F7','beta-F3','beta-Fz','beta-F4','beta-F8',
             'beta-P3','beta-Pz','beta-P4','beta-PO7','beta-PO8', 
             'theta-F7','theta-F3','theta-Fz','theta-F4','theta-F8',
             'theta-P3','theta-Pz','theta-P4','theta-PO7','theta-PO8'] 
node_names_orig = ['F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'Cz', 'T8', 'P3', 'Pz', 'P4', 'PO7', 'PO8', 'Oz']

# Generate all possible combination pairs of nodes
combinations_total = plv_names + node_names_orig*3

alpha_plv, beta_plv, theta_plv = [], [], []
alpha_spec, beta_spec, theta_spec = [], [], []
for i in indexes:
    if i < 10:
        alpha_plv.append(combinations_total[i])
    elif i >= 10 and i < 20:
        beta_plv.append(combinations_total[i])
    elif i >= 20 and i < 30:
        theta_plv.append(combinations_total[i])
    elif i >= 30 and i < 44:
        alpha_spec.append(combinations_total[i])
    elif i >= 44 and i < 58:
        beta_spec.append(combinations_total[i])
    elif i >= 58 and i < 72:
        theta_spec.append(combinations_total[i])
        
# print(alpha_comb, beta_comb, theta_comb)
print('PLV: ', len(alpha_plv), len(beta_plv), len(theta_plv))
# print(alpha_spec, beta_spec, theta_spec)
print('Spectrum: ', len(alpha_spec), len(beta_spec), len(theta_spec))
    
frontal_count, parietal_count = 0, 0
channel_counts = {}
frequency_bands = [alpha_spec, beta_spec, theta_spec]
for frequency_band in frequency_bands:
    for channel in frequency_band:
        # if 'F' in channel:
        #     frontal_count += 1
        # elif 'P' in channel:
        #     parietal_count += 1
        if channel in channel_counts:
            channel_counts[channel] += 1
        else:
            channel_counts[channel] = 1
    
cirkel_diagram(channel_counts, model='Model8')
barplot_features(channel_counts, frequency_bands, node_names_orig, model='Model8')

chosen_frequency_bands8 = [alpha_spec, beta_spec, theta_spec]

frontal_count, parietal_count = 0, 0
channel_counts = {}
frequency_bands = [alpha_plv, beta_plv, theta_plv]
for frequency_band in frequency_bands:
    for channel in frequency_band:
        # if 'F' in channel:
        #     frontal_count += 1
        # elif 'P' in channel:
        #     parietal_count += 1
        if channel in channel_counts:
            channel_counts[channel] += 1
        else:
            channel_counts[channel] = 1
    
barplot_PLVfeatures(channel_counts, frequency_bands, plv_names, model='Model8')

chosen_plv8 = [alpha_plv, beta_plv, theta_plv]


# Model 9 - RFE on selected spectral features (from baseline) + all 30 frontal-parietal PLV features

In [ ]:
# loading data 
directory = '/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'
# directory = '/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'

concatenated_X, concatenated_Y = [], []

for subdir in sorted(os.listdir(directory))[1:]: # index from 1 because Mac has .DS_Store file
    for file in sorted(os.listdir(os.path.join(directory, subdir))):
        data = np.load(os.path.join(directory, subdir, file))
        X_PLV = frontal_parietal_PLVs(data['X'][:135]) # 30 PLV features (alpha, beta, theta)
        X_Freq = data['X'][indexes_baseline] # by baseline selected frequency spectrum features
        X = np.concatenate((X_PLV, X_Freq))
        Y = data['Y']
        concatenated_X.append(X)
        concatenated_Y.append(Y)

concatenated_Y = np.concatenate(concatenated_Y)
    

In [ ]:
# SVM + RFE

X = concatenated_X
Y = concatenated_Y

# Create an SVM classifier
classifier = svm.SVC(random_state=42)

# Set the hyperparameters to tune
# parameters = {'C': [0.1, 0.3, 1, 5, 10, 16, 30, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': [0.00001, 0.0005, 0.005, 0.01, 0.05, 0.15, 1, 3, 6]}
parameters = {'C': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.2, 1.5, 1.7, 2, 2.2, 2.5, 2.7, 3, 3.5, 4, 4.5, 5, 6, 7, 8, 9, 10, 12, 15, 17, 20, 22, 25, 27, 30, 35, 40, 45, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': ['auto', 'scale', 0.000001, 0.000002, 0.000003, 0.000004, 0.000005, 0.00001, 0.00002, 0.00003, 0.00004, 0.00005, 0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.001, 0.002, 0.003, 0.004, 0.005, 0.01, 0.02, 0.03, 0.04, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.6, 0.8, 1, 2, 3, 5, 6, 8, 10, 12, 15, 17, 20, 22, 25, 27, 30]}

# Define the scoring metric
scoring = make_scorer(f1_score, average='macro')

# Set the number of folds for cross-validation
num_folds = 10

# Create the Stratified k-fold object
skf = StratifiedKFold(n_splits=num_folds, shuffle=False)

f1_scores, accuracy_scores, precision_scores, recall_scores, cm_scores, best_parameters_list = [],[],[],[],[],[]
k_values = list(range(1, num_folds + 1))

# Perform stratified cross-validation
for k, (train_index, test_index) in enumerate(skf.split(X, Y), start=1):
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    Y_train, Y_test = Y[train_index], Y[test_index]
    
    # Perform feature selection using RFE with LinearSVC as the estimator
    estimator = svm.LinearSVC(max_iter=30000)  # Use LinearSVC as the estimator for feature selection
    selector = RFE(estimator)
    selector.fit(X_train, Y_train)
    X_train_selected = selector.transform(X_train)
    X_test_selected = selector.transform(X_test)
    
    # Get the selected feature indices
    selected_feature_indices = selector.support_
    selected_feature_ranking = selector.ranking_

    # Use grid search to tune the hyperparameters with F1-score as the scoring metric
    grid_search = GridSearchCV(classifier, parameters, scoring=scoring)
    grid_search.fit(X_train_selected, Y_train)

    # Get the best hyperparameters
    best_parameters = grid_search.best_params_

    # Train the classifier using the best hyperparameters
    classifier = svm.SVC(kernel=best_parameters['kernel'], C=best_parameters['C'], gamma=best_parameters['gamma'])
    classifier.fit(X_train_selected, Y_train)

    # Make predictions on the test set
    Y_pred = classifier.predict(X_test_selected)

    # Calculater performance metrics and retrieve best parameter set
    best_parameters_list.append(best_parameters) 
    f1_scores.append(f1_score(Y_test, Y_pred, average='macro'))
    accuracy_scores.append(accuracy_score(Y_test, Y_pred))
    precision_scores.append(precision_score(Y_test, Y_pred, average='macro', zero_division=1))
    recall_scores.append(recall_score(Y_test, Y_pred, average='macro'))
    cm_scores.append(confusion_matrix(Y_test, Y_pred))
    
    # Print the progress
    f1 = f1_score(Y_test, Y_pred, average='macro')
    print(f'Fold {k} - F1-score: {round(f1, 2)} - Best Parameters: {best_parameters}')
    
# print the models performance metrices and its best parameter set
print(' ')
print('Mean F1-score:', f'{round(np.mean(f1_scores), 2)}.', 'Std F1-score:', f'{round(np.std(f1_scores), 2)}.','Max F1-score:', round(np.max(f1_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(f1_scores)]}')
print(' ')
print('Mean Accuracy-score:', f'{round(np.mean(accuracy_scores), 2)}.', 'Std Accuracy-score:', f'{round(np.std(accuracy_scores), 2)}.', 'Max Accuracy-score:', round(np.max(accuracy_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(accuracy_scores)]}')
print(' ')
print('Mean Precision-score:', f'{round(np.mean(precision_scores), 2)}.', 'Std Precision-score:', f'{round(np.std(precision_scores), 2)}.', 'Max Precision-score:', round(np.max(precision_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(precision_scores)]}')
print(' ')
print('Mean Recall-score:', f'{round(np.mean(recall_scores), 2)}.', 'Std Recall-score:', f'{round(np.std(recall_scores), 2)}.', 'Max Recall-score:', round(np.max(recall_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(recall_scores)]}')
print(' ')

elbow_plot(k_values, f1_scores, model='Model9')
cm_plot(cm_scores, f1_scores, model='Model9', total=True)

model9_f1 = f1_scores
model9_param = best_parameters_list[np.argmax(f1_scores)]

In [ ]:
# features selected by the model

# features selected by the model
indexes = indexes_baseline
node_names_orig = ['F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'Cz', 'T8', 'P3', 'Pz', 'P4', 'PO7', 'PO8', 'Oz']
combinations_total = node_names_orig*3
alpha_spec, beta_spec, theta_spec = [], [], []
for i in indexes:
    if i < 14:
        alpha_spec.append(combinations_total[i])
    elif i >= 14 and i < 28:
        beta_spec.append(combinations_total[i])
    elif i >= 28 and i < 42:
        theta_spec.append(combinations_total[i])

# Get the indexes where the array is True
indexes = np.where(selected_feature_indices)[0]
plv_names = ['alpha-F7','alpha-F3','alpha-Fz','alpha-F4','alpha-F8',
             'alpha-P3','alpha-Pz','alpha-P4','alpha-PO7','alpha-PO8',
             'beta-F7','beta-F3','beta-Fz','beta-F4','beta-F8',
             'beta-P3','beta-Pz','beta-P4','beta-PO7','beta-PO8', 
             'theta-F7','theta-F3','theta-Fz','theta-F4','theta-F8',
             'theta-P3','theta-Pz','theta-P4','theta-PO7','theta-PO8'] 
# # Generate all possible combination pairs of nodes
combinations_total = plv_names + alpha_spec + beta_spec + theta_spec
alpha_plv, beta_plv, theta_plv = [], [], []
alpha_spec1, beta_spec1, theta_spec1 = [], [], []
for i in indexes:
    if i < 10:
        alpha_plv.append(combinations_total[i])
    elif i >= 10 and i < 20:
        beta_plv.append(combinations_total[i])
    elif i >= 20 and i < 30:
        theta_plv.append(combinations_total[i])
    elif i >= 30 and i < 30+len(alpha_spec):
        alpha_spec1.append(combinations_total[i])
    elif i >= 30+len(alpha_spec) and i < 30+len(alpha_spec)+len(beta_spec):
        beta_spec1.append(combinations_total[i])
    elif i >= 30+len(alpha_spec)+len(beta_spec) and i < 30+len(alpha_spec)+len(beta_spec)+len(theta_spec):
        theta_spec1.append(combinations_total[i])
        
print('PLV: ', len(alpha_plv), len(beta_plv), len(theta_plv))
# print(alpha_spec, beta_spec, theta_spec)
print('Spectrum: ', len(alpha_spec1), len(beta_spec1), len(theta_spec1))
    
frontal_count, parietal_count = 0, 0
channel_counts = {}
frequency_bands = [alpha_spec1, beta_spec1, theta_spec1]
for frequency_band in frequency_bands:
    for channel in frequency_band:
        # if 'F' in channel:
        #     frontal_count += 1
        # elif 'P' in channel:
        #     parietal_count += 1
        if channel in channel_counts:
            channel_counts[channel] += 1
        else:
            channel_counts[channel] = 1
    
cirkel_diagram(channel_counts, model='Model9')
barplot_features(channel_counts, frequency_bands, node_names_orig, model='Model9')

chosen_frequency_bands9 = [alpha_spec, beta_spec, theta_spec]

frontal_count, parietal_count = 0, 0
channel_counts = {}
frequency_bands = [alpha_plv, beta_plv, theta_plv]
for frequency_band in frequency_bands:
    for frequency in frequency_band:
        if frequency in channel_counts:
            channel_counts[frequency] += 1
        else:
            channel_counts[frequency] = 1
    
barplot_PLVfeatures(channel_counts, frequency_bands, plv_names, model='Model9')

chosen_plv9 = [alpha_plv, beta_plv, theta_plv]


# Model 10 - selected spectral features (from baseline) + RFE on all 30 frontal-parietal PLV features

In [ ]:
# loading data 
directory = '/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'
# directory = '/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'

concatenated_X, concatenated_Y = [], []

for subdir in sorted(os.listdir(directory))[1:]: # index from 1 because Mac has .DS_Store file
    for file in sorted(os.listdir(os.path.join(directory, subdir))):
        data = np.load(os.path.join(directory, subdir, file))
        X_PLV = frontal_parietal_PLVs(data['X'][:135]) # 30 PLV features (10 alpha, 10 beta, 10 theta)
        X_Freq = data['X'][indexes_baseline] # by baseline selected frequency spectrum features
        X = np.concatenate((X_PLV, X_Freq))
        Y = data['Y']
        concatenated_X.append(X)
        concatenated_Y.append(Y)

concatenated_Y = np.concatenate(concatenated_Y)
    

In [ ]:
# SVM + RFE

X = concatenated_X
Y = concatenated_Y

# Create an SVM classifier
classifier = svm.SVC(random_state=42)

# Set the hyperparameters to tune
parameters = {'C': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.2, 1.5, 1.7, 2, 2.2, 2.5, 2.7, 3, 3.5, 4, 4.5, 5, 6, 7, 8, 9, 10, 12, 15, 17, 20, 22, 25, 27, 30, 35, 40, 45, 50], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': ['auto', 'scale', 0.000001, 0.000002, 0.000003, 0.000004, 0.000005, 0.00001, 0.00002, 0.00003, 0.00004, 0.00005, 0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.001, 0.002, 0.003, 0.004, 0.005, 0.01, 0.02, 0.03, 0.04, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.6, 0.8, 1, 2, 3, 5, 6, 8, 10, 12, 15, 17, 20, 22, 25, 27, 30]}

# Define the scoring metric
scoring = make_scorer(f1_score, average='macro')

# Set the number of folds for cross-validation
num_folds = 10

# Create the Stratified k-fold object
skf = StratifiedKFold(n_splits=num_folds, shuffle=False)

f1_scores, accuracy_scores, precision_scores, recall_scores, cm_scores, best_parameters_list = [],[],[],[],[],[]
k_values = list(range(1, num_folds + 1))

# Perform stratified cross-validation
for k, (train_index, test_index) in enumerate(skf.split(X, Y), start=1):
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    Y_train, Y_test = Y[train_index], Y[test_index]
    
    # Perform feature selection using RFE with LinearSVC as the estimator
    estimator = svm.LinearSVC(max_iter=30000)  # Use LinearSVC as the estimator for feature selection
    selector = RFE(estimator)
    # Select the PLV features in X_train and X_test
    X_train_plv = X_train[:, :30]
    X_test_plv = X_test[:, :30]

    # Perform feature selection on the selected features
    selector.fit(X_train_plv, Y_train)

    # Get the selected feature indices
    selected_feature_indices = selector.support_
    selected_feature_ranking = selector.ranking_

    # Apply feature selection on X_train and X_test
    X_train_plv_selected = X_train_plv[:, selected_feature_indices]
    X_test_plv_selected = X_test_plv[:, selected_feature_indices]

    # Concatenate the remaining features with the selected features
    X_train_selected = np.concatenate((X_train_plv_selected, X_train[:, 30:]), axis=1)
    X_test_selected = np.concatenate((X_test_plv_selected, X_test[:, 30:]), axis=1)

    # Use grid search to tune the hyperparameters with F1-score as the scoring metric
    grid_search = GridSearchCV(classifier, parameters, scoring=scoring)
    grid_search.fit(X_train_selected, Y_train)

    # Get the best hyperparameters
    best_parameters = grid_search.best_params_

    # Train the classifier using the best hyperparameters
    classifier = svm.SVC(kernel=best_parameters['kernel'], C=best_parameters['C'], gamma=best_parameters['gamma'])
    classifier.fit(X_train_selected, Y_train)

    # Make predictions on the test set
    Y_pred = classifier.predict(X_test_selected)

    # Calculater performance metrics and retrieve best parameter set
    best_parameters_list.append(best_parameters) 
    f1_scores.append(f1_score(Y_test, Y_pred, average='macro'))
    accuracy_scores.append(accuracy_score(Y_test, Y_pred))
    precision_scores.append(precision_score(Y_test, Y_pred, average='macro', zero_division=1))
    recall_scores.append(recall_score(Y_test, Y_pred, average='macro'))
    cm_scores.append(confusion_matrix(Y_test, Y_pred))
    
    # Print the progress
    f1 = f1_score(Y_test, Y_pred, average='macro')
    print(f'Fold {k} - F1-score: {round(f1, 2)} - Best Parameters: {best_parameters}')
    
# print the models performance metrices and its best parameter set
print(' ')
print('Mean F1-score:', f'{round(np.mean(f1_scores), 2)}.', 'Std F1-score:', f'{round(np.std(f1_scores), 2)}.','Max F1-score:', round(np.max(f1_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(f1_scores)]}')
print(' ')
print('Mean Accuracy-score:', f'{round(np.mean(accuracy_scores), 2)}.', 'Std Accuracy-score:', f'{round(np.std(accuracy_scores), 2)}.', 'Max Accuracy-score:', round(np.max(accuracy_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(accuracy_scores)]}')
print(' ')
print('Mean Precision-score:', f'{round(np.mean(precision_scores), 2)}.', 'Std Precision-score:', f'{round(np.std(precision_scores), 2)}.', 'Max Precision-score:', round(np.max(precision_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(precision_scores)]}')
print(' ')
print('Mean Recall-score:', f'{round(np.mean(recall_scores), 2)}.', 'Std Recall-score:', f'{round(np.std(recall_scores), 2)}.', 'Max Recall-score:', round(np.max(recall_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(recall_scores)]}')
print(' ')

elbow_plot(k_values, f1_scores, model='Model10')
cm_plot(cm_scores, f1_scores, model='Model10', total=True)

model10_f1 = f1_scores
model10_param = best_parameters_list[np.argmax(f1_scores)]

In [ ]:
# features selected by the model

# Get the indexes where the array is True
indexes = np.where(selected_feature_indices)[0]

plv_names = ['alpha-F7','alpha-F3','alpha-Fz','alpha-F4','alpha-F8',
             'alpha-P3','alpha-Pz','alpha-P4','alpha-PO7','alpha-PO8',
             'beta-F7','beta-F3','beta-Fz','beta-F4','beta-F8',
             'beta-P3','beta-Pz','beta-P4','beta-PO7','beta-PO8', 
             'theta-F7','theta-F3','theta-Fz','theta-F4','theta-F8',
             'theta-P3','theta-Pz','theta-P4','theta-PO7','theta-PO8'] 
node_names_orig = ['F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'Cz', 'T8', 'P3', 'Pz', 'P4', 'PO7', 'PO8', 'Oz']
tot = node_names_orig*3

alpha_plv, beta_plv, theta_plv = [], [], []
for i in indexes:
    if i < 10:
        alpha_plv.append(plv_names[i])
    elif i >= 10 and i < 20:
        beta_plv.append(plv_names[i])
    elif i >= 20 and i < 30:
        theta_plv.append(plv_names[i])
        
alpha_spec, beta_spec, theta_spec = [], [], []
for j in indexes_baseline:
    if j < 14:
        alpha_spec.append(tot[j])
    elif j >= 14 and j < 28:
        beta_spec.append(tot[j])
    elif j >= 28 and j < 42:
        theta_spec.append(tot[j])
        
# print(alpha_comb, beta_comb, theta_comb)
print('PLV: ', len(alpha_plv), len(beta_plv), len(theta_plv))
    
frontal_count, parietal_count = 0, 0
channel_counts = {}
frequency_bands = [alpha_spec, beta_spec, theta_spec]
for frequency_band in frequency_bands:
    for channel in frequency_band:
        # if 'F' in channel:
        #     frontal_count += 1
        # elif 'P' in channel:
        #     parietal_count += 1
        if channel in channel_counts:
            channel_counts[channel] += 1
        else:
            channel_counts[channel] = 1
    
# cirkel_diagram(channel_counts, model='Model10')
# barplot_features(channel_counts, frequency_bands, node_names_orig, model='Model10')

chosen_frequency_bands10 = [alpha_spec, beta_spec, theta_spec]

frontal_count, parietal_count = 0, 0
channel_counts = {}
frequency_bands = [alpha_plv, beta_plv, theta_plv]
for frequency_band in frequency_bands:
    for channel in frequency_band:
        if 'F' in channel:
            frontal_count += 1
        elif 'P' in channel:
            parietal_count += 1
        if channel in channel_counts:
            channel_counts[channel] += 1
        else:
            channel_counts[channel] = 1
    
barplot_PLVfeatures(channel_counts, frequency_bands, plv_names, model='Model10')

chosen_plv10 = [alpha_plv, beta_plv, theta_plv]


# Final plots

In [ ]:
# Boxplot f1-scores per model.

# List of f1 scores for each model
f1_scorers = [raw_model_f1, baseline_model_f1, model1_f1, model2_f1, model3_f1, model4_f1, model5_f1, model6_f1, model7_f1, model8_f1, model9_f1, model10_f1]

# Create a figure and axis
fig, ax = plt.subplots(figsize=(8, 6))

# Define colors for each model
colors = ['blue', 'green', 'orange', 'purple', 'pink', 'cyan', 'yellow', 'brown', 'grey', 'teal', 'magenta', 'gold']

# Create the boxplot with vertical bars and individual colors
bp = ax.boxplot(f1_scorers, patch_artist=True, notch=True, vert=True)

# Set facecolor and edgecolor for each box
for box, color in zip(bp['boxes'], colors):
    box.set(facecolor=color, edgecolor='black')

# Set colors for whiskers, caps, and medians
for whisker, cap, median, color in zip(bp['whiskers'], bp['caps'], bp['medians'], colors):
    whisker.set(color='black')
    cap.set(color='black')
    median.set(color='red')  # Set the median line color to red

# Set labels and title
ax.set_xticklabels(['Raw', 'Baseline', 'Model 1', 'Model 2', 'Model 3', 'Model 4', 'Model 5', 'Model 6', 'Model 7', 'Model 8', 'Model 9', 'Model 10'], size=8)
ax.set_ylabel('F1 Score')
ax.set_title('F1 Scores of each model')

# Add mean markers
means = [np.mean(f1) for f1 in f1_scorers]
ax.plot(range(1, len(f1_scorers) + 1), means, marker='o', color='red', linestyle='None', label='Mean')

# Add standard deviation markers
stds = [np.std(f1) for f1 in f1_scorers]
ax.errorbar(range(1, len(f1_scorers) + 1), means, yerr=stds, color='black', linestyle='None', capsize=5, label='Std')

# Add legend
ax.legend()

# Display the plot
plt.show()

# Save the plot as an image file
output_file = '/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/boxplot_f1.png'
# output_file = '/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/boxplot_f1.png'
plt.savefig(output_file, bbox_inches='tight')


In [ ]:
# Average connectivity values among participants.

directory = '/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'
# directory = '/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'

concatenated_X, concatenated_Y = [], []

for subdir in sorted(os.listdir(directory))[1:]: # index from 1 because Mac has .DS_Store file
    for file in sorted(os.listdir(os.path.join(directory, subdir))):
        data = np.load(os.path.join(directory, subdir, file))
        X_3 = frontal_parietal_PLV(data['X'][:135]) # 3 PLV features (alpha, beta, theta)
        X_30 = frontal_parietal_PLVs(data['X'][:135]) # 30 PLV features (10 alpha, 10 beta, 10 theta)
        X = np.concatenate((X_3, X_30))
        concatenated_X.append(X)
        
# Calculate the averages for each variable
averages = np.mean(concatenated_X, axis=0)

# Create a figure and axis object
fig, ax = plt.subplots(figsize=(8, 6))

# Create a dot plot for the first variable
dot1 = ax.plot(averages[:1], marker='o', color='red', linestyle='None', markersize=10, markeredgecolor='black', markeredgewidth=1)

# Create a dot plot for the second variable
dot2 = ax.plot(averages[1:2], marker='o', color='green', linestyle='None', markersize=10, markeredgecolor='black', markeredgewidth=1)

# Create a dot plot for the third variable
dot3 = ax.plot(averages[2:3], marker='o', color='blue', linestyle='None', markersize=10, markeredgecolor='black', markeredgewidth=1)

# Create a line plot for the fourth 10 variables
line1, = ax.plot(np.concatenate((np.array([None]), averages[3:13])), color='red', linewidth=1.5)

# Create a line plot for the fifth 10 variables
line2, = ax.plot(np.concatenate((np.array([None]), averages[13:23])), color='green', linewidth=1.5)

# Create a line plot for the sixth 10 variables
line3, = ax.plot(np.concatenate((np.array([None]), averages[23:33])), color='blue', linewidth=1.5)

# Set the x-axis label
ax.set_xlabel('Connectivity pairs')

# Set the y-axis label
ax.set_ylabel('Average')

# Create a legend
legend_labels = {
    tuple(dot1): 'Alpha_all',
    tuple(dot2): 'Beta_all',
    tuple(dot3): 'Theta_all',
    line1: 'Alpha',
    line2: 'Beta',
    line3: 'Theta'
}

plv_names = ['total PLV', 'F7','F3','Fz','F4','F8','P3','Pz','P4','PO7','PO8'] 
ax.set_xticks(range(len(plv_names)), [])
ax.set_xticklabels(plv_names, rotation=45, ha='right', fontsize=13)
ax.legend(legend_labels.keys(), legend_labels.values())

# Show the plot
plt.show()

# Save the plot as an image file
output_file = '/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/averageconnectivityplot.png'
# output_file = '/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/averageconnectivityplot.png'
plt.savefig(output_file, bbox_inches='tight')

# Clear the current figure
plt.close()

In [ ]:
# Boxplot connectivity values among participants.

# Create a figure and axis object
fig, ax = plt.subplots(figsize=(8, 6))

# Define the color groups
colors = ['red', 'green', 'blue','red', 'green', 'blue']

# Iterate over each variable in the arrays
for i in range(len(concatenated_X[0])):
    # Determine the color group for the current variable
    if i < 3: 
       color_index = i
    else:
        color_index = ((i-3) // 10) + 3  # Grouping every 10 variables

    color = colors[color_index]

    # Extract the data for the current variable
    variable_data = [array[i] for array in concatenated_X]

    # Create a boxplot for the current variable with the assigned color
    box = ax.boxplot(variable_data, positions=[i], boxprops={'color': color}, patch_artist=True)
    
    # Set the facecolor of the boxes
    for patch in box['boxes']:
        patch.set_facecolor(color)
        
# Set the labels for each boxplot
ax.set_position([0.1, 0.1, 1.9, 0.7])  # Adjust the values as desired 
ax.set_xticks(range(len(concatenated_X[0])))
plv_names = ['alpha-PLV', 'beta-PLV', 'theta-PLV', 'alpha-F7','alpha-F3','alpha-Fz','alpha-F4','alpha-F8',
             'alpha-P3','alpha-Pz','alpha-P4','alpha-PO7','alpha-PO8',
             'beta-F7','beta-F3','beta-Fz','beta-F4','beta-F8',
             'beta-P3','beta-Pz','beta-P4','beta-PO7','beta-PO8', 
             'theta-F7','theta-F3','theta-Fz','theta-F4','theta-F8',
             'theta-P3','theta-Pz','theta-P4','theta-PO7','theta-PO8'] 
ax.set_xticklabels(plv_names, rotation=45, ha='right', fontsize=13)
ax.set_title('Boxplots connectivity (PLV) measurements for each channel and the frequency bands', fontsize=20)
# Set the y-axis label
ax.set_ylabel('Value')

# Show the plot
plt.show()

# Save the plot as an image file
output_file = '/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/boxplot_connectivity.png'
# output_file = '/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/boxplot_connectivity.png'
plt.savefig(output_file, bbox_inches='tight')

# Clear the current figure
plt.close()

In [ ]:
# Custom plot functions for selected features over all models: spectral features barplot and PLV features barplot.

def custom_spectralfeatures_barplot(channel_counts, frequency_bands, all_channels, model):

    # # Extract unique channels and their counts
    # unique_channels = list(channel_counts.keys())

    # # Sort unique channels based on custom order
    # custom_order = ['F', 'O', 'T', 'C', 'P']
    # unique_channels = sorted(unique_channels, key=lambda x: (custom_order.index(x[0]), x))

    # Assign colors to each frequency band
    colors = ['r', 'g', 'b']

    # Plotting the stacked barplot
    x = np.arange(len(all_channels))
    
    fig, ax = plt.subplots(figsize=(8, 6))
    bottom = np.zeros(len(all_channels))

    for i, freq in zip(range(len(frequency_bands)), ['Alpha', 'Beta', 'Theta']):
        for frequency in frequency_bands:
            values = [channel_counts[channel] if channel in frequency[i] else 0 for channel in all_channels]
            ax.bar(x, values, bottom=bottom, color=colors[i])
            bottom += values
    
    ax.set_xlabel('Channels chosen accross frequency ranges', fontsize=10)
    ax.set_title(f'Chosen spectral features by {model}', fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels(all_channels, rotation=45, ha='right', fontsize=8)
    ax.yaxis.set_ticks([])  # Remove ticks on y-axis
    
    # Create custom legend manually
    legend_labels = ['Alpha', 'Beta', 'Theta']
    legend_colors = ['r', 'b', 'g']
    legend_patches = [mpatches.Patch(color=color, label=label) for color, label in zip(legend_colors, legend_labels)]
    ax.legend(handles=legend_patches)

    plt.show()
    
    # Save the plot as an image file
    output_file = f'/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/barplotspectrumfeatures_{model}.png'
    # output_file = f'/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/barplotspectrumfeatures_{model}.png'
    plt.savefig(output_file, bbox_inches='tight')
    
    # Clear the current figure
    plt.clf()
    
def custom_PLVfeatures_barplot(channel_counts, frequency_bands, all_channels, model):

    # # Extract unique channels and their counts
    # unique_channels = list(channel_counts.keys())

    # temp_list = []
    # if 'alpha-PLV' in unique_channels:
    #     temp_list.append('Alpha_allPLV')
    # if 'beta-PLV' in unique_channels:
    #     temp_list.append('Beta_allPLV')
    # if 'theta-PLV' in unique_channels:
    #     temp_list.append('Theta_allPLV')

    # Assign colors to each frequency band
    colors = ['r', 'g', 'b', 'r', 'g', 'b']

    # Plotting the stacked barplot
    x = np.arange(len(all_channels))

    fig, ax = plt.subplots(figsize=(8, 6))
    bottom = np.zeros(len(all_channels))
    
    for freq in frequency_bands[:2]:
        for i in range(3):
            values = [channel_counts[channel] if channel in freq[i] else 0 for channel in all_channels]
            ax.bar(x, values, bottom=bottom, color=colors[i])
            bottom += values
        
    for i, freq in zip(range(len(frequency_bands[2:])), ['Alpha', 'Beta', 'Theta']):
        for frequency in frequency_bands[2:]:
            values = [channel_counts[channel] if channel in frequency[i] else 0 for channel in all_channels]
            ax.bar(x, values, bottom=bottom, color=colors[i])
            bottom += values
    
    # Adjust the position and size of the axes
    ax.set_position([0.1, 0.1, 0.9, 0.3])  # Adjust the values as desired  
    ax.set_xlabel('PLV feature per channel and the frequency ranges', fontsize=10)
    ax.set_title(f'Chosen PLV features by {model}', fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels(all_channels, rotation=45, ha='right', fontsize=8)
    ax.yaxis.set_ticks([])  # Remove ticks on y-axis
    
    plt.show()
    
    # Save the plot as an image file
    output_file = f'/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/barplotPLVfeatures_{model}.png'
    # output_file = f'/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/barplotPLVfeatures_{model}.png'
    plt.savefig(output_file, bbox_inches='tight')
    
    # Clear the current figure
    plt.clf()


In [ ]:
# Plotting custom plots for selected features over all models.

plv_names = ['alpha-PLV','beta-PLV','theta-PLV','alpha-F7','alpha-F3','alpha-Fz','alpha-F4','alpha-F8',
             'alpha-P3','alpha-Pz','alpha-P4','alpha-PO7','alpha-PO8',
             'beta-F7','beta-F3','beta-Fz','beta-F4','beta-F8',
             'beta-P3','beta-Pz','beta-P4','beta-PO7','beta-PO8', 
             'theta-F7','theta-F3','theta-Fz','theta-F4','theta-F8',
             'theta-P3','theta-Pz','theta-P4','theta-PO7','theta-PO8'] 
node_names_orig = ['F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'Cz', 'T8', 'P3', 'Pz', 'P4', 'PO7', 'PO8', 'Oz']
    
frontal_count, parietal_count = 0, 0
channel_counts = {}
chosen_FREQUENCYBANDs = chosen_frequency_bands,chosen_frequency_bands3,chosen_frequency_bands4,chosen_frequency_bands8,chosen_frequency_bands9
frequency_bands = chosen_FREQUENCYBANDs
for model in frequency_bands:
    for frequency_band in model:
        for channel in frequency_band:
            if channel in channel_counts:
                channel_counts[channel] += 1
            else:
                channel_counts[channel] = 1
            
custom_spectralfeatures_barplot(channel_counts, frequency_bands, node_names_orig, model='all_models')

frontal_count, parietal_count = 0, 0
channel_counts = {}
chosen_PLVs = chosen_plv3, chosen_plv4, chosen_plv5, chosen_plv8, chosen_plv9, chosen_plv10
frequency_bands = chosen_PLVs
for model in frequency_bands:
    for frequency_band in model:
        for channel in frequency_band:
            if channel in channel_counts:
                channel_counts[channel] += 1
            else:
                channel_counts[channel] = 1

custom_PLVfeatures_barplot(channel_counts, frequency_bands, plv_names, model='all_models')


In [ ]:
# Selected spectral features and PLV features per model.

chosen_PLVs = chosen_plv3, chosen_plv4, chosen_plv5, chosen_plv8, chosen_plv9, chosen_plv10

models = ['Baseline', 'Model 3', 'Model 4', 'Model 8', 'Model 9']
for name, selected in zip(models, chosen_FREQUENCYBANDs):
    print(f'Selected spectral features of model {name}: {selected}')

models = ['Model 3', 'Model 4', 'Model 5', 'Model 8', 'Model 9', 'Model 10']
for name, selected in zip(models, chosen_PLVs):
    print(f'Selected PLV features of model {name}: {selected}') 

In [ ]:
# Plot distribution of selected SVM-hyperparameters.

from collections import Counter
import matplotlib.pyplot as plt

chosen_param = raw_model_param, baseline_model_param, model1_param, model2_param, model3_param, model4_param, model5_param, model6_param, model7_param, model8_param, model9_param, model10_param

# Count the occurrences of each hyperparameter value
c_counts = Counter(param['C'] for param in chosen_param)
gamma_counts = Counter(param['gamma'] for param in chosen_param)
kernel_counts = Counter(param['kernel'] for param in chosen_param)

# Plot pie charts for each hyperparameter
def plot_hyperparameter_distribution(counts, title, model):
    labels, values = zip(*counts.items())
    plt.pie(values, labels=labels, autopct='%1.1f%%')
    plt.title(title)
    plt.axis('equal')
    plt.show()
    
    # Save the plot as an image file
    output_file = f'/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/selected_hyperparameters_SVM/hyperparameters_{model}.png'
    # output_file = f'/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/selected_hyperparameters_SVM/hyperparameters_{model}.png'
    plt.savefig(output_file, bbox_inches='tight')
    
    # Clear the current figure
    plt.clf()

plot_hyperparameter_distribution(c_counts, 'Distribution of C values', model='C')
plot_hyperparameter_distribution(gamma_counts, 'Distribution of gamma values', model='gamma')
plot_hyperparameter_distribution(kernel_counts, 'Distribution of kernel values', model='kernel')

In [ ]:
# Plot best performing model with highest accuracy on fold.

new_list = []
for i in f1_scorers:
    idx = np.argmax(i)
    new_list.append(i[idx])
idx1 = np.argmax(new_list)

if idx1 == 0:
    model = 'Raw' 
elif idx1 == 1:
    model = 'Baseline'
elif idx1 >= 2:
    model = idx1-1
print(f'Best performing model {model} \n with a max f1 score of {new_list[idx1]}. \n with a mean f1 score of {np.mean(f1_scorers[idx1])}. \n its best performing set of hyperparameters: {chosen_param[idx1]}.')

In [ ]:
# Plot best performing model with highest accuracy on average.

mean_f1_scorers = np.mean(f1_scorers, axis=1)
idx = np.argmax(mean_f1_scorers)
idx1 = np.argmax(f1_scorers[idx])
if idx == 0:
    model = 'Raw'
elif idx == 1:
    model = 'Baseline'
elif idx >= 2:
    model = idx-1
print(f'Highest performing model on average is model {model} \n with a max f1 score of {f1_scorers[idx][idx1]}. \n with a mean f1 score of {np.mean(mean_f1_scorers[idx])}. \n its best performing set of hyperparameters: {chosen_param[idx]}.')

In [ ]:
# Plot average connectivity among samples.

directory = '/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'

concatenated_X_3, concatenated_X_30 = [], []

for subdir in sorted(os.listdir(directory))[1:]: # index from 1 because Mac has .DS_Store file
    for file in sorted(os.listdir(os.path.join(directory, subdir))):
        data = np.load(os.path.join(directory, subdir, file))
        
        X_3 = frontal_parietal_PLV(data['X'][:135])
        av_alpha = X_3[:1]
        av_beta = X_3[1:2]
        av_theta = X_3[2:3]
        X_3_av = np.concatenate(( av_alpha, av_beta, av_theta ))
        concatenated_X_3.append(np.mean(X_3_av))
        
        X_30 = frontal_parietal_PLVs(data['X'][:135])
        av_alpha = np.mean(X_30[:10])
        av_beta = np.mean(X_30[10:20])
        av_theta = np.mean(X_30[20:30])
        X_30_av = np.concatenate(( np.array([av_alpha]), np.array([av_beta]), np.array([av_theta]) ))
        concatenated_X_30.append(np.mean(X_30_av))

# Convert the list to a NumPy array
concatenated_X_30 = np.array(concatenated_X_30)

# Calculate the total width for each group
bar_width = 0.6

# Calculate the offset for each group
offset = np.arange(len(concatenated_X_30))

# Calculate the overall mean
overall_mean = np.mean(concatenated_X_30, axis=0)

# Create a figure and axis object
fig, ax = plt.subplots()

# Plot the grouped bars
ax.bar(offset + bar_width, concatenated_X_30, width=bar_width, color='red')

# Plot the overall mean as a horizontal line
ax.axhline(overall_mean, color='black', linestyle='--', label='Overall Mean')

# Set the y-axis limits
ax.set_ylim(0, 1)

ax.set_position([0.1,0.1,2.5,1.6])
ax.set_title('Average of 30 PLV values among each participant', fontsize=18)

# Set the x-axis ticks and labels
ax.set_xticks(offset + bar_width)
ax.set_xticklabels(range(1, len(concatenated_X_30) + 1), fontsize=6)

# Set the y-axis label
ax.set_ylabel('PLV', fontsize=18)

# Show the plot
plt.show()

# Save the plot as an image file
output_file = f'/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/connectivitydistributionamongparticipants_30PLV.png'
# output_file = f'/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/connectivitydistributionamongparticipants_30PLV.png'
plt.savefig(output_file, bbox_inches='tight')

# Clear the current figure
plt.clf()
